# Library Inclusion and Utilities

<!-- structured-notebook -->
## Notebook Summary
Purpose: document an earlier ProQuest cleaning workflow, including full-text cleanup, title cleanup, relevance scoring, and a long manual audit trail of topic keep/drop decisions before exporting a cleaned dataset.

Main steps:
- Load an older processed ProQuest snapshot and clean the article text fields.
- Compute relevance ratios that help rank noisy topics for manual review.
- Walk through topic-specific decisions in audit-log form, then export the cleaned dataframe.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


In [1]:
import pandas as pd, re
import itables.options as opt
from itables import show
from src.news.nlp import *
from src.news.data_processor import *

/home/iskender/anaconda3/envs/bertopic-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

## Load Pre-built Data

In [2]:
df = pd.read_csv(PROQUEST_PROCESSED_DIR / 'df_v3.csv')

# Cleaning

## Clean Full Text

***
Delete articles with non-available full text
***

In [3]:
col = 'Full text'
mask = (df[col].isna()) | (df[col].str.strip() == '') | (df[col].str.startswith('Not available'))
df[mask]
df = df[~mask]

***
Average amount of words per article
***

In [4]:
total_l = 0
for title in df['Full text']:
    total_l += len(title)
print(total_l/len(df))

4380.29849528644


***
Cleaning "\n" and "\"
***

In [5]:
df['Full text'] = df['Full text'].str.replace("\n", " ")

***
Tokenizing sentences
***

In [6]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /home/iskender/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/iskender/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
df['last'] = [' '.join(sent_tokenize(text)[-4:]) for text in df['Full text']]

***
Cleaning the last sentence part
***

In [8]:
df_dedup = df.copy()

In [9]:
df_dedup['last'].tolist()

['Info Box:How to sign up for ICL spring classes Registration begins Jan. 3 and classes begin Jan. 10. When: Monday through Thursday 9:30 a.m. to 2:30 p.m. Where: The ICL office located at 658 E. 200 South. Cost: $40 for the entire year. Can also register online at www.dixie.edu/com/icl.',
 'A wheel chair, walkers, hospital bed, and other items are available to borrow. Memorial donations received during December were: From Toby Young, in memory of Marie Wright; from Barbara Greenough, in memory of Alice and Harold Shattuck and Fran Streeter; from Paul and Shirley Schofield, in memory of Pat Jennison; from Catherine Kurkul, in memory of Christopher Kurkul; from Dr. Ralph, in memory of Pat Jennison; from Barbara Angers, in memory of Harold Angers; from Peggy and Dirk Jager, in memory of Patty Vigevano; from Rebecca and William Noyes, in memory of Pat Jennison; from Verna and Donald Newcomb, in memory of Leslie and Avis Dodge; from Irene and Jeffrey Michaud, in memory of Gert French; from

## Clean Title

In [10]:
df['Title']

0                           ICL registration begins Jan. 3
1                   Westminster Cares announces activities
2                                        dancing longevity
3                         Healthier future for Gen Y girls
4                           Women's health 'short changed'
                               ...                        
11097    United States: Aging may change some brain cel...
11098                 What Is The Ideal Breakfast Formula?
11099    Stay Calm, Sleep, and 5 More Tips for a Long a...
11100                         SIMPLE STEPS for a good LIFE
11101                WHAT you NEED TO DO FOR A LONGER LIFE
Name: Title, Length: 11032, dtype: object

In [11]:
[title for title in df['Title'] if len(title) < 10]

['The Diary',
 '[Diary ]',
 'Events',
 'Cafe chat',
 'Calendar',
 'Town Talk',
 'Purple',
 'Health',
 'Notices',
 'tv talk',
 'Activity',
 'Sponsor',
 "What's On",
 'BRIEFCASE',
 'Homehunt',
 'Calendar',
 'SENIORS',
 'Briefs',
 'DIGEST',
 'try these',
 'Briefly',
 'Club News',
 'IN BRIEF',
 'LEARN',
 'DURBAN',
 'TIPS',
 "What's On",
 'IN BRIEF',
 'Near you',
 'SENIORS',
 'Reunions',
 "What's On",
 'Try these',
 'DATEBOOK',
 "What's On",
 '3',
 'LOOK OUT',
 'YOUR SAY',
 'Monday/28',
 'Briefly',
 'WERNER',
 'Briefs',
 'Q & A',
 'calendar',
 'COLUMNIST',
 'Club News',
 'ACRES',
 'LETTERS',
 'IN BRIEF',
 'Signpost',
 'Briefly',
 'Players',
 'Events',
 'FYI',
 'IN BRIEF',
 'calendar',
 'PLAYERS',
 "WHAT'S ON",
 'brief',
 'LETTERS',
 'YOUR VIEW',
 'Clipboard',
 'PLAYERS',
 'Diary',
 'Calendar',
 'GO!',
 'BIRTHDAYS',
 'FYI',
 'Community',
 'What’s on',
 'Diet 16:8',
 'BIRTHDAYS',
 'FYI',
 'Calendar',
 'Clipboard',
 'hjlfuil',
 'Community',
 'PLAYERS',
 'BRIEFS',
 'PLAYERS',
 'Aging',
 'FYI',


***
The average length of a title is 64 words.
***

## Relevance Ratio

In [12]:
def find_relevance_ratio(group_df:pd.DataFrame, by: list[str] = ['Title', 'Abstract', 'Full text']):
    keywords = [
        "healthy aging", "healthy ageing",
        "anti-aging", "anti-ageing", "anti aging", "anti ageing",
        "biohacking", "living longer", "health rejuvenation", "nootropics",
        "well aging", "well ageing", "aging well", "ageing well",
        "slow aging", "slow ageing"
    ]
    pattern = "|".join(keywords)
    
    relevant = 0
    for index, record in group_df.iterrows():
        full_str = ".".join([record[col] for col in by]).lower()
        if re.search(pattern, full_str):
            relevant += 1
    group_df['rel_ratio'] = relevant/len(group_df)

    return group_df

In [13]:
df = (
    df.groupby("topic", group_keys=False)
      .apply(find_relevance_ratio)
      .reset_index(drop=True)
)

/tmp/ipykernel_14457/1970550472.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(find_relevance_ratio)


In [14]:
df['rel_bins'] = pd.cut(df['rel_ratio'], [0.2,0.4,0.6,0.8,1], ['Low', 'Average', 'Moderate', 'High'])

In [15]:
show(df.groupby('topic')['rel_ratio'].mean())

Loading ITables v2.4.4 from the internet... (need help?)


<!-- structured-notebook -->
## Manual Topic Review Log
The long sequence below is best read as an audit trail. Topics are grouped by relevance-ratio bands, inspected manually, and then either removed entirely or kept with explicit exception IDs.


## Topic Cleaning

In [16]:
def remove_topic(df:pd.DataFrame, topic_num:int, exceptions:list[int]):
    mask = (df['topic'] == topic_num) & (~df.index.isin(exceptions))
    print(f"The orig length:{len(df)} -> cropped: {len(df[~mask])}")
    return df[~mask]

### Topic rel_ratio 0.2 - 0.4

In [17]:
df[(df["rel_ratio"] > 0.2) & (df["rel_ratio"] <= 0.4)]['topic'].unique()

array([114., 196.,  94.])

#### Topic 114

***
Topic keywords
***

In [18]:
df[df['topic']==114]['topic_keywords'].unique()

array(["['season', 'players', 'hes', 'game', 'strengths', 'coach', 'league', 'games', 'football', 'win', 'louis', 'player', 'guy', 'played', 'teams', 'ryan', 'field', 'knee', 'probably', 'surgery', 'era', 'manager', 'line', 'summer', 'knows', 'man', 'heat', 'runs', 'super', 'return']"],
      dtype=object)

In [19]:
df[df['topic']==114]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
125,"Hossa is aging well: Veteran winger healthy,...","Along with an equally hot Patrick Sharp, who c...",Jonathan Toews has seen Blackhawks linemate Ma...,Athletes; Professional hockey -- Chicago Black...,"Chicago Tribune; Chicago, Ill.",2010,"Oct 20, 2010",Chicago Sports,"Tribune Publishing Company, LLC","Chicago, Ill.",...,Feature,759143597,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,114.0,0.063781,"['season', 'players', 'hes', 'game', 'strength...",After he was moved out of the Hawks' GM slot i...,0.344828,"(0.2, 0.4]"
421,Healthy Garnett lifts Celtics: Vet delivers ...,Boston Celtics coach Doc Rivers has joked in t...,BOSTON -- Boston Celtics coach Doc Rivers has ...,Athletes; Tournaments & championships; Profess...,"USA TODAY; McLean, Va.",2010,"May 14, 2010\n\nDateline: BOSTON",SPORTS,"USA Today, a division of Gannett Satellite Inf...","McLean, Va.",...,News,276456059,https://www.proquest.com/globalnewsfed/newspap...,2022-10-12,114.0,0.214336,"['season', 'players', 'hes', 'game', 'strength...","After the game, James stuck around on the cour...",0.344828,"(0.2, 0.4]"
490,Cox's last stand may be fruitful: Braves focus...,"""There's going to come a point in time where t...",Braves players wore T-shirts beneath their jer...,Coaches & managers; Professional baseball -- A...,"The Atlanta Journal - Constitution; Atlanta, Ga.",2010,"Apr 5, 2010",Sports,"Atlanta Journal Constitution, LLC","Atlanta, Ga.",...,Feature,337696423,https://www.proquest.com/globalnewsfed/newspap...,2017-11-07,114.0,1.000000,"['season', 'players', 'hes', 'game', 'strength...","Jason Heyward, RF 8. Nate McLouth, CF 9. Derek...",0.344828,"(0.2, 0.4]"
586,"NFL:: Free agency, draft offer hope for '10;...","Strengths: QB Matt Ryan, WR Roddy White, agele...","MIAMI, Fla. - The Super Bowl's over, Drew Bree...",NaN,"The Charleston Gazette; Charleston, W.V.\n\nFi...",2010,"Feb 10, 2010",Sports,Charleston Newspapers,"Charleston, W.V.",...,News,331732159,https://www.proquest.com/globalnewsfed/newspap...,2017-11-05,114.0,1.000000,"['season', 'players', 'hes', 'game', 'strength...",Strengths: CB Nnamdi Asomugha; kicking game wi...,0.344828,"(0.2, 0.4]"
629,Todd makes FSC proud: Alum helps coach Hawks,"""It's easy for all of us to second guess and b...",Ray Allen can still hear former Washington Bul...,Coaches & managers; Surgery; Professional bask...,"Telegram & Gazette; Worcester, Mass.\n\nFirst ...",2010,"Jan 17, 2010\n\ncolumn: NBA",SPORTS,"GateHouse Media, Inc.","Worcester, Mass.",...,News,269045258,https://www.proquest.com/globalnewsfed/newspap...,2025-08-08,114.0,0.058438,"['season', 'players', 'hes', 'game', 'strength...",KG has deep pockets Davis joked he would colle...,0.344828,"(0.2, 0.4]"
724,"NO LUCK, SO MAYBE WAITIN' FOR PEYTON?","Oh, he's hurt. He's 35. He's expensive. His ov...",Here's my only problem with being out front on...,NaN,"South Florida Sun - Sentinel; Fort Lauderdale,...",2011,"Nov 16, 2011\n\ncolumn: DAVE HYDE Commentary",Sports,"Tribune Publishing Company, LLC","Fort Lauderdale, Fla.",...,COLUMN,904229285,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,114.0,1.000000,"['season', 'players', 'hes', 'game', 'strength...",But some good can still come out a Dolphins se...,0.344828,"(0.2, 0.4]"
808,Red Wings talent outweighs issues,An aging group of core players cant stay healt...,An aging group of core players cant stay healt...,Athletes\n\nCompany / organization: Name: Detr...,"Washington Examiner; Washington, D.C.\n\nFirst...",2011,"Oct 14, 2011",Sports,"Clarity Media Group, Inc.","Washington, D.C.",...,News,898326906,https://www.proquest.com/globalnewsfed/newspap...,2022-11-11,114.0,1.000000,"['season', 'players', 'hes', 'game', 'strength...",With stars like Henrik Zette

Leave documents: 4171, 6059

In [20]:
df = remove_topic(df, 114, [4171,6059])

The orig length:11032 -> cropped: 11003


#### Topic 196

***
Topic keywords
***

In [21]:
df[df['topic']==196]['topic_keywords'].unique()

array(["['lifestyle factors', 'alzheimers', 'risk developing', 'cognitive', 'healthy lifestyle', 'reduce risk', 'alzheimers disease', 'smoking', 'alcohol', 'risk dementia', 'lower risk', 'blood pressure', 'brains', 'evidence', 'mediterranean', 'brain health', 'high blood', 'cholesterol', 'combination', 'habits', 'decline', 'review', 'followed', 'study researchers', 'cent', 'map', 'behaviors', 'met', 'cognitive decline', 'memory']"],
      dtype=object)

In [22]:
df[df['topic']==196]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
542,Learning keeps brain healthy: [1 ],None available.,"Washington, March. 3 -- Learning keeps the bra...",NaN,The Hindustan Times; New Delhi,2010,"Mar 3, 2010\n\nDateline: Washington",NaN,HT Digital Streams Limited,New Delhi,...,WIRE FEED,471209938,https://www.proquest.com/globalnewsfed/newspap...,2018-02-25,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...",Study results appear in the early online editi...,0.235294,"(0.2, 0.4]"
856,Healthy living could help fight dementia,"According to Dr. Kenneth Rockwood, MD, a profe...",Many people are apprehensive about getting old...,Dementia; Brain; Medical research; Cardiovascu...,"Prince George Citizen; Prince George, B.C.\n\n...",2011,"Sep 27, 2011",Seniors,Postmedia Network Inc.,"Prince George, B.C.",...,News,894789111,https://www.proquest.com/globalnewsfed/newspap...,2023-10-09,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...",Reading is a way to stimulate vocabulary and a...,0.235294,"(0.2, 0.4]"
2416,Let the mind games begin: WELLBEING,Exercise could help reduce the risks of develo...,Exercise could help reduce the risks of develo...,Brain; Physical fitness; Exercise; Age; Dement...,"Sydney Morning Herald; Sydney, N.S.W.\n\nFirst...",2013,"Aug 3, 2013",Life,Fairfax Digital,"Sydney, N.S.W.",...,News,1416802993,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...","""The implication is that environmental factors...",0.235294,"(0.2, 0.4]"
2863,Simple ways to reduce Alzheimer's risks,"The charity's review, which included the lates...",Lifestyle is responsible for about three-quart...,Dementia; Older people; Studies; Lifestyles; A...,Irish Examiner; Cork,2014,"Dec 16, 2014",Ireland,The Examiner Echo Group Ltd.,Cork,...,News,1636453180,https://www.proquest.com/newsp apers/simple-wa...,2023-10-09,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...",But the evidence review also showed that a hea...,0.235294,"(0.2, 0.4]"
2864,"Follow five golden rules to prevent dementia, ...","According to the latest estimates, there are 8...",Lifestyle is responsible for more than three q...,Older people; Studies; Dementia; Exercise; Alz...,Telegraph.co.uk; London,2014,"Dec 15, 2014",News,Telegraph Media Group Holdings Limited,London,...,News,1636342640,https://www.proquest.com/newspapers/follow-fiv...,2023-10-09,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...",It's now time for these messages to start reac...,0.235294,"(0.2, 0.4]"
2866,Follow four golden rules to ward off dementia:...,"According to the latest estimates, there are 8...",FOLLOWING four out of five golden rules for he...,Older people; Studies; Hypertension; Age; Phys...,The Daily Telegraph; London (UK)\n\nFirst page: 1,2014,"Dec 15, 2014",News; Front Page,Daily Telegraph,London (UK),...,News,1636249243,https://www.proquest.com/newspapers/follow-fou...,2023-12-06,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...","""and keeping your cholesterol in check."" Howev...",0.235294,"(0.2, 0.4]"
3907,United States: A Healthy Body Often Equals a H...,"By keeping the brain both healthy and active, ...",People who want to stay sharp as they age ofte...,Aging; Dementia; Brain; Diet; Alzheimer's dise...,Asia News Monitor; Bangkok,2015,"Jul 8, 2015",Health News,Thai News Service Group,Bangkok,...,News,1694542447,https://www.proquest.com/globalnewsfed/newspap...,2024-12-04,196.0,1.0,"['lifestyle factors', 'alzheimers', 'risk deve...","Do something that requires thought, whether th...",0.235294,"(0.2, 0.4]"
4828,United States: A Healthy Heart May Protect an ...,"""The results of our study highlight the need f...",New research adds to a growing body of evidenc...,Cholesterol; Older people; M

#### Topic 94

***
Topic keywords
***

In [23]:
df[df['topic']==94]['topic_keywords'].unique()

array(["['australia', 'healthy brain', 'people dementia', 'syndrome', 'centre healthy', 'people living', 'estimated', 'alzheimers', 'readers', 'rice', 'id', 'memory', 'henry', 'awareness', 'billion', 'number people', 'street', 'september', 'supported', 'appeal', 'john', 'australians', '2050', '2025', 'executive officer', 'van', 'dedicated', 'encourage', 'projected', 'prevalence']"],
      dtype=object)

In [24]:
df[df['topic']==94]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
923,Hoarding can be a serious disease,"Then there is the worst form of hoarding, know...","Q: Mymother, now in her 80s, is starting to ho...",NaN,"The Spectator; Hamilton, Ont.\n\nFirst page: GO.5",2011,"Sep 2, 2011",Go,"Torstar Syndication Services, a Division of To...","Hamilton, Ont.",...,News,886948564,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,94.0,0.061603,"['australia', 'healthy brain', 'people dementi...",But I would first recommend that you have your...,0.375,"(0.2, 0.4]"
929,To plan and prepare for this change,To plan and prepare for this change in demogra...,To plan and prepare for this change in demogra...,Dementia,"Port Macquarie News; Port Macquarie, N.S.W.\n\...",2011,"Aug 31, 2011",NaN,Rural Press Pty Ltd Australian Community Media...,"Port Macquarie, N.S.W.",...,News,885865551,https://www.proquest.com/globalnewsfed/newspap...,2011-08-30,94.0,1.000000,"['australia', 'healthy brain', 'people dementi...",North Coast Institute of TAFE head teacher of ...,0.375,"(0.2, 0.4]"
1460,"Prostate group hears about dementia, healthy a...",The Murray Bridge Prostate Cancer Support Grou...,The Murray Bridge Prostate Cancer Support Grou...,Older people; Meetings,"The Murray Valley Standard; Murray Bridge, S. ...",2012,"Nov 13, 2012",NaN,Rural Press Pty Ltd Australian Community Media...,"Murray Bridge, S. Aust.",...,News,1151095597,https://www.proquest.com/glob alnewsfed/newspa...,2012-11-12,94.0,0.103003,"['australia', 'healthy brain', 'people dementi...",Key elements to help find things were motivati...,0.375,"(0.2, 0.4]"
3019,Dementia-friendly nation,"DURING Dementia Awareness Month in September, ...","DURING Dementia Awareness Month in September, ...",Dementia; Alzheimer's disease,"Bay Post; Batemans Bay, N.S.W.",2014,"Sep 25, 2014",NaN,Rural Press Pty Ltd Australian Community Media...,"Batemans Bay, N.S.W.",...,News,1565566994,https://www.proquest.com/newspapers/dementia-f...,2023-10-06,94.0,1.000000,"['australia', 'healthy brain', 'people dementi...",Greater awareness and understanding of dementi...,0.375,"(0.2, 0.4]"
3037,Dementia-friendly future,During Dementia Awareness Month throughout Sep...,During Dementia Awareness Month throughout Sep...,Dementia; Inner city,"Illawarra Mercury; Wollongong, N.S.W.\n\nFirst...",2014,"Sep 15, 2014",NaN,Rural Press Pty Ltd Australian Community Media...,"Wollongong, N.S.W.",...,News,1561938209,https://www.proquest.com/newspapers/dementia-f...,2017-11-22,94.0,0.210679,"['australia', 'healthy brain', 'people dementi...",The tour along various inner city streets woul...,0.375,"(0.2, 0.4]"
3051,Fast response appreciated: opinion YOUR SAY,"Fast response appreciated LAST Sunday, a two-y...","Fast response appreciated LAST Sunday, a two-y...",Dementia\n\nLocation: Australia; United States...,"Newcastle Herald; Newcastle, N.S.W.\n\nFirst p...",2014,"Sep 13, 2014",Letters,Rural Press Pty Ltd Australian Community Media...,"Newcastle, N.S.W.",...,News,1561787894,https://www.proquest.com/newspapers/fast-respo...,2014-09-14,94.0,1.000000,"['australia', 'healthy brain', 'people dementi...",Newcastle is growing up under its own steam an...,0.375,"(0.2, 0.4]"
3175,Dementia project for Kiama,UNIVERSITY of Wollongong researchers will see ...,UNIVERSITY of Wollongong researchers will see ...,Dementia; Older people; Age; Design; Alzheimer...,"Illawarra Mercury; Wollongong, N.S.W.\n\nFirst...",2014,"Jun 27, 2014",NaN,Rural Press Pty Ltd Australian Community Media...,"Wollongong, N.S.W.",...,News,1540647878,https://www.proquest.com/newspapers/dementia-p...,2023-10-09,94.0,1.000000,"['australia', 'healthy brain', 'people dementi...",Dr Phillipson said the research would lead to ...,0.375,"(0.2, 0.4]"
3182,Second health event at practice,None availab

### Topic rel_ratio 0.4 - 0.6

In [25]:
df[(df["rel_ratio"] > 0.4) & (df["rel_ratio"] <= 0.6)]['topic'].unique()

array([158., 136., 214.])

#### Topic 158

In [26]:
df[df['topic']==158]['topic_keywords'].unique()

array(["['dance', 'dancing', 'classes', 'music', 'youve', 'youve got', 'margaret', 'moves', 'australian', 'sydney', 'line', 'class', 'physical activity', 'fun', 'mood', 'popular', 'floor', 'health wellbeing', 'felt', 'love', 'began', 'happiness', 'examining', 'stage', 'happy', 'kelly', 'balance', 'professional', 'husband', 'sharp']"],
      dtype=object)

In [27]:
df[df['topic']==158]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
2,dancing longevity,"""It's good both physically and mentally becaus...","For instructor, is key to Thelma Nelson believ...",NaN,"South Bend Tribune; South Bend, Ind.\n\nFirst ...",2010,"Dec 30, 2010",NaN,South Bend Tribune Corporation,"South Bend, Ind.",...,News,821827067,https://www.proquest.com/globalnewsfed/newspap...,2011-02-07,158.0,0.541152,"['dance', 'dancing', 'classes', 'music', 'youv...","""All it takes is music to get me tapping."" In ...",0.571429,"(0.4, 0.6]"
657,A DANCE TO THE MUSIC OF TIME,Company budgets mostly don't stretch to assist...,Sometimes retirement comes too early. Jane Alb...,Ballet; Dance; Retirement\n\nCompany / organiz...,"Weekend Australian; Canberra, A.C.T.\n\nFirst ...",2011,"Dec 31, 2011",Review,Nationwide News Pty Ltd,"Canberra, A.C.T.",...,News,913038388,https://www.proquest.com/globalnewsfed/newspap...,2011-12-30,158.0,1.000000,"['dance', 'dancing', 'classes', 'music', 'youv...","""I don't miss the day-to-day slog, I don't mis...",0.571429,"(0.4, 0.6]"
2682,Area News in Brief: Seniors can learn health...,None available.,The Long Beach Island Health Department is hos...,NaN,"Asbury Park Press; Asbury Park, N.J.\n\nFirst ...",2013,"Mar 21, 2013",A,Gannett Media Corp,"Asbury Park, N.J.",...,News,1321088862,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,158.0,0.018672,"['dance', 'dancing', 'classes', 'music', 'youv...","For more information, go to www.noyesmuseum.or...",0.571429,"(0.4, 0.6]"
3528,Is this the reel cure for old age?: Scots co...,"Elizabeth Foster, executive officer of the Roy...",IT is known and loved worldwide for its boiste...,Physical fitness; Dance; Studies,Mail on Sunday; London (UK)\n\nFirst page: 48,2014,"Jan 12, 2014",News,"Solo Syndication, a division of Associated New...",London (UK),...,News,1476559717,https://www.proquest.com/newspapers/is-this-re...,2014-01-12,158.0,1.000000,"['dance', 'dancing', 'classes', 'music', 'youv...","Elizabeth Foster, executive officer of the Roy...",0.571429,"(0.4, 0.6]"
3852,"Happy, healthy & fit golden years: Dance gat...","""I have to wear high heels,"" the Yakima woman ...",Sue MacMichaels is 99 years old and still danc...,NaN,"Yakima Herald - Republic; Yakima, Wash.\n\nFir...",2015,"Jul 30, 2015",NaN,Yakima Herald Republic,"Yakima, Wash.",...,News,1701982265,https://www.proquest.com/globalnewsfed/newspap...,2015-08-06,158.0,1.000000,"['dance', 'dancing', 'classes', 'music', 'youv...","They're my inspiration,"" Cooper said. ""I'm hop...",0.571429,"(0.4, 0.6]"
4040,news briefs: [2 ],ANDOVERGet your broken stuff fixed at this cli...,ANDOVERGet your broken stuff fixed at this cli...,NaN,"Star Tribune; Minneapolis, Minn.\n\nFirst page...",2015,"Apr 29, 2015",NEWS,Star Tribune Media Company LLC,"Minneapolis, Minn.",...,News,1676473091,https://www.proquest.com/globalnewsfed/newspap...,2017-11-22,158.0,0.008890,"['dance', 'dancing', 'classes', 'music', 'youv...",Those who are interested may attend two meetin...,0.571429,"(0.4, 0.6]"
5126,Dance your way to a healthier ageing brain,None available.,NYT Syndicate Dance classes may beat tradition...,Researchers; Exercise; Physical fitness; Older...,Qatar Tribune; Doha,2017,"Oct 17, 2017",NaN,"Disco Digital Media, Inc.",Doha,...,News,1951389576,https://www.proquest.com/globalnewsfed/newspap...,2023-10-09,158.0,1.000000,"['dance', 'dancing', 'classes', 'music', 'youv...",But he agreed that exercise in general plus a ...,0.571429,"(0.4, 0.6]"
5689,Health benefits of dancing for the elderly,None available.,"Pakistan, Dec. 1 -- ""Dance, when you are broke...",Population; Older people; Dance\n\nLocation: P...,Daily Times; Lahore,2018,"Dec 1, 2018\n\nDateline: Pakistan",NaN,HT Digital Streams Limited,Lahore,...,News,2139684210,https://www.proquest.

Noisy articles: 4080, 2716

In [28]:
df = df[~df.index.isin([4080,2716])]

#### Topic 34

In [29]:
df[df['topic']==34]['topic_keywords'].unique()

array(["['company organization', 'naics', 'laura', 'texas', 'jones', 'md', 'susan', 'linda', 'cross', 'text', 'edition', 'julie', 'print', 'don', 'document', 'jr', 'pat', 'stephen', 'ann', 'nancy', 'eric', 'jean', 'display', 'lisa', 'jennifer', 'william', 'kim', 'jim', 'carol', 'tom']"],
      dtype=object)

In [30]:
df[df['topic']==34]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
1682,High-protein diet crucial for healthy aging [F...,None available.,Display of this document is currently not perm...,NaN,"The Province; Vancouver, B.C.",2012,"Aug 10, 2012",NaN,Postmedia Network Inc.,"Vancouver, B.C.",...,News,1033574815,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,34.0,0.010055,"['company organization', 'naics', 'laura', 'te...",Display of this document is currently not perm...,1.0,"(0.8, 1.0]"


***
Let's drop these since Full text is not available for them. If we need more data later, we can search it.
***

In [31]:
df = remove_topic(df,34,[])

The orig length:11001 -> cropped: 11000


#### Topic 136

In [32]:
df[df['topic']==136]['topic_keywords'].unique()

array(["['games', 'game', 'video', 'cognitive', 'memory', 'played', 'computer', 'playing', 'mild', 'cognition', 'skills', 'cognitive impairment', 'functioning', 'impairment', 'visual', 'app', 'alzheimers', 'cognitive function', 'fitness', 'memory loss', 'function', 'grace', 'san francisco', 'journal', 'francisco', 'brains', 'tasks', 'prof', 'performed', 'san']"],
      dtype=object)

In [33]:
df[df['topic']==136]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
96,EVERYONE'S TALKING ABOUT SUCCESSFUL AGING ACTI...,"According to the U.S. Census Bureau, 35 millio...",Note: SPECIAL SECTION: SUCCESSFUL AGING HEALTH...,Aging; Older people; Colleges & universities; ...,"Daily News; Los Angeles, Calif.\n\nFirst page:...",2010,"Nov 4, 2010",L.A. Life,Los Angeles Newspaper Group,"Los Angeles, Calif.",...,News,762401249,https://www.proquest.com/globalnewsfed/newspap...,2023-10-09,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...","Where: Sheraton Universal Hotel, Universal Cit...",0.56,"(0.4, 0.6]"
200,Mind games: Does play really help the brain?,"""Even if brain-fitness programs can improve th...","Crosswords, Sudoku, Nintendo - now scientists ...",NaN,"The Globe and Mail; Toronto, Ont.",2010,"Sep 22, 2010",National News,The Globe and Mail,"Toronto, Ont.",...,News,751959679,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...","""Even if brain-fitness programs can improve th...",0.56,"(0.4, 0.6]"
1300,Higher levels of social activity cut the risk ...,None available.,"Washington, Feb. 18 -- Higher levels of social...",NaN,The Hindustan Times; New Delhi,2011,"Feb 18, 2011\n\nDateline: Washington",NaN,HT Digital Streams Limited,New Delhi,...,WIRE FEED,852626739,https://www.proquest.com/globalnewsfed/newspap...,2018-02-26,136.0,0.064593,"['games', 'game', 'video', 'cognitive', 'memor...",Future research is needed to determine whether...,0.56,"(0.4, 0.6]"
1879,Can video games promote healthier aging?,Video game technology has significant benefits...,"Berlin, April 26 -- Video game technology has ...",Older people; Social interaction; Journals\n\n...,The South Asian Times; Hicksville,2012,"Apr 26, 2012\n\nDateline: Berlin",NaN,HT Digital Streams Limited,Hicksville,...,NEWSPAPER,1010045636,https://www.proquest.com/globalnewsfed/newspap...,2012-04-28,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...",The second study was conducted in three Europe...,0.56,"(0.4, 0.6]"
2300,HEALTHY LIVING: Computer games exercise aging ...,Because of the success of high-tech memory pro...,If you're worried about age-related memory los...,Physical fitness; Memory; Brain research; Olde...,"The Atlanta Journal - Constitution; Atlanta, Ga.",2013,"Sep 29, 2013",FEATURES,"Atlanta Journal Constitution, LLC","Atlanta, Ga.",...,Feature,1437433298,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...",> Engage in social activities that emphasize i...,0.56,"(0.4, 0.6]"
2349,Play for your brain,A VIDEO game can help elderly people fight cog...,A VIDEO game can help elderly people fight cog...,Older people; Medical research,"Herald Sun; Melbourne, Vic.\n\nFirst page: 32",2013,"Sep 6, 2013",GENERALNEW,Nationwide News Pty Ltd,"Melbourne, Vic.",...,News,142984 9063,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...","The 3D game had hidden complexities, pushing p...",0.56,"(0.4, 0.6]"
2355,Video games may boost brain power,Studies have shown that specially-made games c...,Video games may seem like a mindless way to wa...,Older people; Computer & video games; Children...,"The Poughkeepsie Journal; Poughkeepsie, N.Y.\n...",2013,"Sep 8, 2013",E,Gannett Media Corp,"Poughkeepsie, N.Y.",...,News,1430760286,https://www.proquest.com/globalnewsfed/newspap...,2019-12-06,136.0,1.000000,"['games', 'game', 'video', 'cognitive', 'memor...",Forcing people to keep driving up windy mounta...,0.56,"(0.4, 0.6]"
2356,Study Finds Video Games Aid Memory In Elderly,"""Challenging yourself with new situations is n...","SAN FRANCISCO - Video games, largely considere...

#### Topic 214

In [34]:
df[df['topic']==214]['topic_keywords'].unique()

array(["['workshops', 'ohio', 'living healthy', 'located', 'chronic health', 'workshop', 'manage', 'medical center', 'chronic conditions', 'arthritis', 'stanford', 'chronic disease', 'calling', 'score', 'confidence', 'elder', '300', 'condition', 'avenue', 'health conditions', 'options', 'metro', 'leaders', 'register', 'bureau', 'healthy living', 'class', 'managing', 'officer', 'stroke']"],
      dtype=object)

In [35]:
df[df['topic']==214]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
131,Aging gracefully: Program that helps seniors...,"""Take Charge of your Health: Live Well, Be Wel...",CHAMPAIGN - A program that helps seniors - or ...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: A.1",2010,"Oct 16, 2010",NaN,News Gazette,"Champaign, Ill.",...,News,761077019,https://www.proquest.com/globalnewsfed/newspap...,2016-08-21,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...","""Seeing renewed energy, hope and confidence re...",0.533333,"(0.4, 0.6]"
1044,Edgewood Fine Arts Academy hosts free Healthy ...,"On Friday, July 29, Spanishspeaking San Antoni...","On Friday, July 29, Spanishspeaking San Antoni...",Meetings; Hispanic Americans; Health; Aging; C...,"La Prensa; San Antonio, Tex.\n\nVolume: 23\n\n...",2011,"Jul 10, 2011",NaN,La Prensa San Antonio,"San Antonio, Tex.",...,News,878898524,https://www.proquest.com/globalnewsfed/newspap...,2013-12-31,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...",There will be a thorough explanation of differ...,0.533333,"(0.4, 0.6]"
1047,Elder Options offers help with chronic illnesses,"""Living Healthy is a program that works,"" said...",Are you looking for ways to manage your chroni...,NaN,"Gainesville Sun; Gainesville, Fla.",2011,"Jul 7, 2011",NaN,Halifax Media Group,"Gainesville, Fla.",...,News,875247783,https://www.proquest.com/globalnewsfed/newspap...,2012-10-12,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...",They have experience they can share and it wor...,0.533333,"(0.4, 0.6]"
1514,PIH Health is the communities' health and well...,"Better Choices, Better Health : This evidence-...","The ""pay it forward"" concept - the idea that i...",NaN,"Whittier Daily News; Whittier, Calif.",2012,"Oct 19, 2012",News,Los Angeles Newspaper Group,"Whittier, Calif.",...,News,1113477241,https://www.proquest.com/globalnewsfed/newspap...,2012-10-20,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...","To learn how you can ""pay it forward,"" contact...",0.533333,"(0.4, 0.6]"
1592,St. Nicholas offers help for living with chron...,Living Well is a component of the Wisconsin In...,How do you put life back in your life when you...,Older people; Hospitals\n\nCompany / organizat...,"Press; Sheboygan, Wis.",2012,"Sep 19, 2012",Life & Style,Sheboygan Press,"Sheboygan, Wis.",...,News,1080963178,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...",Living Well is a component of the Wisconsin In...,0.533333,"(0.4, 0.6]"
2011,Find ways to live well through workshops,"Living Well might be the workshop for you, if ...",Are you an adult with an ongoing health condit...,Workshops; Aging; Patient education,"Marshfield News Herald; Marshfield, Wis.",2012,"Mar 2, 2012",MNH,"Ogden Newspapers, Inc.","Marshfield, Wis.",...,News,925776949,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...",Spring is an energizing time for renewal of li...,0.533333,"(0.4, 0.6]"
3039,Healthy U free classes kick off Oct. 9,The six-week series will run from Oct. 9 to No...,Do you have a chronic health condition like di...,Cardiovascular disease; Diabetes\n\nCompany / ...,"Cincinnati Enquirer; Cincinnati, Ohio\n\nFirst...",2014,"Sep 18, 2014\n\ncolumn: Hyde Park",S,Gannett Media Corp,"Cincinnati, Ohio",...,News,1563040159,https://www.proquest.com/newspapers/healthy-u-...,2014-09-19,214.0,1.0,"['workshops', 'ohio', 'living healthy', 'locat...","Now, I focus on what I can do to make my life ...",0.533333,"(0.4, 0.6]"
3201,People with chronic conditions needed to share...,"So, [Ronald Jones] signed up for a free progra...","Last year, Ronald Jones decided to get some he...",NaN,"Gainesville Sun; Gainesville, F

### Topic rel_ratio 0.6 - 0.8

In [36]:
df[(df["rel_ratio"] > 0.6) & (df["rel_ratio"] <= 0.8)]['topic'].unique()

array([104.,  97.,  54.,  72.,  26., 177., 178.,  17.,  48., 134.,  67.,
       210., 147.,  20., 118.,   0., 113.,   9., 115.,  47., 188., 201.,
        28., 151., 108., 212., 141.,  30., 164., 122., 140., 174., 219.,
        83.,  81., 216., 217.,  82.,  74., 123., 121.,  12., 160.,  45.,
        70.,  33.,  59., 131.,  62., 222.,   6.,  85., 155., 198., 125.,
       199., 126., 120., 181., 135., 152.])

In [37]:
subset = df[(df["rel_ratio"] > 0.6) & (df["rel_ratio"] <= 0.8)]
for topic, keywords in subset.groupby('topic')['topic_keywords'].unique().items():
    print(topic, "\n", keywords)

0.0 
 ["['skin', 'hair', 'beauty', 'products', 'cream', 'face', 'lines', 'antiaging', 'vitamin', 'ingredients', 'appearance', 'sun', 'antiageing', 'product', 'natural', 'oil', 'youthful', 'treatments', 'damage', 'acid', 'fine', 'antioxidants', 'signs', 'makeup', 'production', 'water', 'pounds', 'contains', 'spots', 'dry']"]
6.0 
 ["['singapore', 'mr', 'programmes', 'programme', 'ageing population', 'cent', 'retirement', 'silver', 'ms', 'government', 'generation', 'residents', 'life expectancy', 'expectancy', 'minister', 'prof', 'workers', 'savings', 'financial', 'ministry', 'costs', 'centres', '2030', 'employees', 'lee', 'encourage', 'aged 65', 'workforce', 'managing', 'retirement age']"]
9.0 
 ["['alzheimers', 'alzheimers disease', 'memory', 'cognitive', 'diagnosis', 'symptoms', 'alzheimers association', 'brains', 'memory loss', 'tests', 'decline', 'clinical', 'brain health', 'impairment', 'test', 'progression', 'healthy brain', 'cognitive impairment', 'risk factors', 'mild', 'risk de

#### Topic 6

In [38]:
df[df['topic']==6]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
747,Govt to shoulder larger burden of medical care...,HEALTH-CARE spending will go up as Singapore a...,HEALTH-CARE spending will go up as Singapore a...,Older people; Hospitals; Health care expenditu...,The Straits Times; Singapore,2011,"Nov 4, 2011",Prime News,SPH Media Limited,Singapore,...,News,902138110,https://www.proquest.com/globalnewsfed/newspap...,2011-11-04,6.0,0.475679,"['singapore', 'mr', 'programmes', 'programme',...",It is not extra-curricular activity. It's part...,0.785124,"(0.6, 0.8]"
802,Aid schemes for those unable to afford nominal...,The Health Promotion Board (HPB) launched the ...,We would like to thank Mrs Lillian Lee for her...,Medical screening,The Straits Times; Singapore,2011,"Oct 16, 2011",Think,SPH Media Limited,Singapore,...,News,898466502,https://www.proquest.com/globalnewsfed/newspap...,2011-10-16,6.0,0.020079,"['singapore', 'mr', 'programmes', 'programme',...","Nevertheless, we believe that, by co-paying fo...",0.785124,"(0.6, 0.8]"
1196,Help for the elderly before they need it: Pl...,"The Tsao Foundation, a non-profit organisation...",MANY elderly people have physical and mental h...,Older people; Health care; Aging; Health servi...,The Straits Times; Singapore,2011,"Apr 16, 2011",Saturday Special Report,SPH Media Limited,Singapore,...,Feature,862133995,https://www.proquest.com/globalnewsfed/newspap...,2011-07-22,6.0,0.321205,"['singapore', 'mr', 'programmes', 'programme',...",So this project will look at issues such as ho...,0.785124,"(0.6, 0.8]"
1197,Ageing with grace: Research is gearing up he...,[...] research is moving away from this model ...,SINGAPORE'S top minds are rallying to tackle a...,Aging; Quality of life; Older people\n\nCompan...,The Straits Times; Singapore,2011,"Apr 16, 2011\n\ncolumn: country for old men",Saturday Special Report,SPH Media Limited,Singapore,...,Feature,862133975,https://www.proquest.com/globalnewsfed/newspap...,2011-07-22,6.0,0.215474,"['singapore', 'mr', 'programmes', 'programme',...","'At this time, health will be the most valuabl...",0.785124,"(0.6, 0.8]"
1243,Silver lining: What are the major challenges...,In order to achieve the goal of self-sufficien...,Wong Su-Yen Senior Partner and Asean Managing ...,Older people; Society; Aging; Stereotypes; Old...,The Business Times; Singapore,2011,"Mar 21, 2011\n\ncolumn: THIS WEEK'S TOPIC",Views From The Top,SPH Media Limited,Singapore,...,News,857946989,https://www.proquest.com/globalnewsfed/newspap...,2024-12-01,6.0,0.042489,"['singapore', 'mr', 'programmes', 'programme',...",Many countries face a similar situation as the...,0.785124,"(0.6, 0.8]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10660,More synergy on the cards for schemes like Hea...,None available.,"In his second term as Health Minister, Mr Ong ...",Walking; Age; Chronic illnesses\n\nLocation: S...,The Straits Times; Singapore,2025,"Jun 2, 2025",Front Page Singapore,SPH Media Limited,Singapore,...,News,3214464016,https://www.proquest.com/globalnewsfed/newspap...,2025-06-02,6.0,0.092555,"['singapore', 'mr', 'programmes', 'programme',...",The information technology operations executiv...,0.785124,"(0.6, 0.8]"
10773,Seniors are taking the kampung spirit beyond t...,None available.,Much is being done to help seniors in Singapor...,Connectivity; Long term health care; Collabora...,The Straits Times; Singapore,2025,"Apr 26, 2025",Front Page Opinion,SPH Media Limited,Singapore,...,News,3194911867,https://www.proquest.com/globalnewsfed/newspap...,2025-04-26,6.0,0.035630,"['singapore', 'mr', 'programmes', 'programme',...",An ageing-in-networks approach allows seniors ...,0.785124,"(0.6, 0.8]"
10888,'Don't suffer in silence': Women leaders worki...,None available.,"By many

#### Topic 20

In [39]:
df[df['topic']==20]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
64,"too good to miss: For nonprofit, charity or ...",Alzheimer Society of Hamilton and Halton Care ...,All numbers are in 905 area code unless otherw...,NaN,"The Spectator; Hamilton, Ont.\n\nFirst page: G...",2010,"Nov 22, 2010",Go,"Torstar Syndication Services, a Division of To...","Hamilton, Ont.",...,News,807637429,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,20.0,1.000000,"['pm', 'wednesday', 'st', 'monday', 'thursday'...","The McQuesten's Childhood Christmas, Nov. 20, ...",0.791667,"(0.6, 0.8]"
147,COUNTY BRIEFS,None available.,"Expo Annual 50+EXPO, presented by the Howard C...",NaN,"The Baltimore Sun; Baltimore, Md.\n\nFirst pag...",2010,"Oct 10, 2010",LOCAL,"Tribune Publishing Company, LLC","Baltimore, Md.",...,News,757292946,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,20.0,0.083322,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",Vendors needed Hickory Ridge Community Associa...,0.791667,"(0.6, 0.8]"
213,Access Transcona offers the following. Call 93...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Access Transcona offers the following. Call 93...,NaN,"Winnipeg Free Press; Winnipeg, Man.\n\nFirst p...",2010,"Sep 17, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,751110174,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,20.0,1.000000,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",The study includes physical assessment of the ...,0.791667,"(0.6, 0.8]"
219,Access East offers the following. Call 938-500...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Access East offers the following. Call 938-500...,NaN,"Winnipeg Free Press; Winnipeg, Man.",2010,"Sep 15, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,750514928,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,20.0,1.000000,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",The study includes physical assessment of the ...,0.791667,"(0.6, 0.8]"
224,Access River East offers the following. Call 9...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Access River East offers the following. Call 9...,NaN,"Winnipeg Free Press; Winnipeg, Man.\n\nFirst p...",2010,"Sep 10, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,750102943,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,20.0,1.000000,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",The study includes physical assessment of the ...,0.791667,"(0.6, 0.8]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8683,'Combat social isolation' 5 questions with Jes...,None available.,5 QUESTIONS WITH | Jess Ray Each week The Pant...,Committees; Recreation; Blood & organ donation...,"Pantagraph; Bloomington, Ill.\n\nFirst page: A2",2022,"Jun 12, 2022",lifestyles/health-med-fit,Pantagraph Publishing Co.,"Bloomington, Ill.",...,News,2698675074,https://www.proquest.com/globalnewsfed/newspap...,2022-08-05,20.0,1.000000,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",What have you done prior to this position? Bef...,0.791667,"(0.6, 0.8]"
8883,Things To Do,None available.,"Wednesday, March 9 ONLINE: Bingocize, 10-11 a....",Bible; Biblical studies; Aging,"Bangor Daily News; Bangor, Me.",2022,"Mar 7, 2022",NaN,Bangor Daily News,"Bangor, Me.",...,News,2637161192,https://www.proquest.com/globalnewsfed/newspap...,2022-03-09,20.0,0.057084,"['pm', 'wednesday', 'st', 'monday', 'thursday'...",All others need to follow the 6 foot distance....,0.791667,"(0.6, 0.8]"
9013,What's Happening,None available.,"Wednesday, Jan. 5 CARIBOU: Miss Erin's Picture...",Support groups; Public libraries; Food; Caregi...,"Bangor Daily News; Bangor, Me.",2022,"Jan 3, 2022",NaN,Bangor Daily News,"Bangor, M

***
This topic mainly contains advertisements
***

#### Topic 28

In [40]:
df[df['topic']==28].head()

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
139,Seniors symposium puts focus on healthy aging,"""We've been lucky, we've had a lot of support ...","For almost two decades, the Seniors' Wellness ...",NaN,"Penticton Western News; Penticton, B.C.\n\nFir...",2010,"Oct 14, 2010",News,Black Press Group Ltd.,"Penticton, B.C.",...,News,758611903,https://www.proquest.com/globalnewsfed/newspap...,2010-10-16,28.0,1.000000,"['expo', 'fair', 'speakers', 'tables', 'talks'...",The Senior's Wellness Society and the South Ok...,0.783333,"(0.6, 0.8]"
142,November health meeting scheduled,Current committees are focused on child obesit...,Special to the News-Record & Sentinel The Madi...,Meetings,"Asheville Citizen - Times; Asheville, N.C.\n\n...",2010,"Oct 13, 2010",News,Gannett Media Corp,"Asheville, N.C.",...,News,1471241710,https ://www.proquest.com/globalnewsfed/newspa...,2017-11-20,28.0,0.176862,"['expo', 'fair', 'speakers', 'tables', 'talks'...",Some who can't attend meetings have contribute...,0.783333,"(0.6, 0.8]"
230,Wellness Expo set for seniors,It will showcase local organizations offering ...,You don't have to be elderly to take a day to ...,NaN,"Tribune; Guelph, Ont.\n\nFirst page: 1",2010,"Sep 9, 2010",News,"Torstar Syndication Services, a Division of To...","Guelph, Ont.",...,News,7501238 75,https://www.proquest.com/globalnewsfed/newspap...,2010-09-10,28.0,1.000000,"['expo', 'fair', 'speakers', 'tables', 'talks'...",A large number of presentations are also sched...,0.783333,"(0.6, 0.8]"
438,HEALTHY LIVING RESOURCES FOR OLDER ADULTS,Programs for older adults including caregiver ...,Visit CITIZEN-TIMES.com for a copy of the Bunc...,Aging; Older people; Swimming; Sports training...,"Asheville Citizen - Times; Asheville, N.C.",2010,"May 9, 2010",News,Gannett Media Corp,"Asheville, N.C.",...,News,275736287,https://www.proquest.com/globalnewsfed/newspap...,2017-11-06,28.0,0.244114,"['expo', 'fair', 'speakers', 'tables', 'talks'...",Register at 225-1904 or www.onecenteryoga.com....,0.783333,"(0.6, 0.8]"
504,Senior Resource Expo connects seniors with var...,"""Spread the word and encourage your friends an...","""Let's keep your walker in the closet and your...",NaN,"Oakland Tribune; Oakland, Calif.",2010,"Mar 24, 2010",My Town,Bay Area News Group,"Oakland, Calif.",...,News,352454723,https://www.proquest.com/globalnewsfed/newspap...,2011-08-24,28.0,0.118034,"['expo', 'fair', 'speakers', 'tables', 'talks'...","""Students come over for dinner with our reside...",0.783333,"(0.6, 0.8]"


***
The topic 28 is mainly about symposium and expos related to healthy aging.
***

#### Topic 48

In [41]:
df[df['topic']==48]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
43,Walking for exercise,The problem was that [Zak] had never been arou...,We discovered walking for exercise before we r...,NaN,"The Weekly Anchor; Edson, Alta.\n\nFirst page: 6",2010,"Nov 29, 2010\n\ncolumn: A Retired Farmer's Alm...",Comment,Postmedia Network Inc.,"Edson, Alta.",...,Column,816798544,https://www.proquest.com/globalnewsfed/newspap...,2014-03-08,48.0,0.284350,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...",Being fit has really enriched our retirement. ...,0.75,"(0.6, 0.8]"
290,SCIENCE FARE,Millions of Americans with disabilities rely o...,"Upcoming science, nature and technology progra...",NaN,"The Santa Fe New Mexican; Santa Fe, N.M.\n\nFi...",2010,"Aug 7, 2010",HEALTH/SCIENCE,Santa Fe New Mexican,"Santa Fe, N.M.",...,News,742442353,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...",Face Facts also will educate viewers about wha...,0.75,"(0.6, 0.8]"
502,Dogs' lives may offer answers to cancer-free l...,"Waters, head of the Gerald P. Murphy Cancer Fo...",Purdue University researcher David Waters hope...,Studies; Cancer; Bone density; Research parks;...,"Daily Record; Morristown, N.J.",2010,"Mar 23, 2010",NaN,Gannett Media Corp,"Morristown, N.J.",...,News,440206150,https://www.proquest.com/globalnewsfed/newspap...,2017-11-16,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...",This is where the fresh ideas on cancer resear...,0.75,"(0.6, 0.8]"
686,Aging cat might have senile dementia,"Again, we had to return because his cough had ...","Dear Dr. Fox:My 18-year-old tabby, Nick, has l...",Vitamin E; Asthma; Weight control; Dementia,"The Washington Post; Washington, D.C.",2011,"Dec 8, 2011",LOCAL LIVING,WP Company LLC d/b/a The Washington Post,"Washington, D.C.",...,News,909167535,https://www.proquest.com/globalnewsfed/newspap...,2023-12-06,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...","Michael W. Fox, author of a newsletter and boo...",0.75,"(0.6, 0.8]"
736,Aging gracefully: Tips to keep your senior d...,Keep on movin': Muscles will atrophy from lack...,Most dog owners consider their pet to be a par...,NaN,"The Spectator; Hamilton, Ont.\n\nFirst page: G.1",2011,"Nov 9, 2011",GO,"Torstar Syndication Services, a Division of To...","Hamilton, Ont.",...,News,902723296,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...",This means don't let your older dog lie around...,0.75,"(0.6, 0.8]"
1338,Globe West People: Tufts veterinarians write t...,"According to Dr. Nicholas Dodman, an animal be...","OLD DOGS, NEW TIPS: Thanks to advances in medi...",Humane societies; Veterinarians; Nonfiction; V...,"Boston Globe; Boston, Mass.",2011,"Jan 23, 2011",West,"Boston Globe Media Partners, LLC","Boston, Mass.",...,News,846998980,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...","That was huge."" For more information, visit ww...",0.75,"(0.6, 0.8]"
1425,Keeping cat claws off the couch,Give your dog water flavored with chicken or b...,Q. Yesterday we adopted a kitten from the Huma...,Cats; Dogs; Diet; Surgery,"Asbury Park Press; Asbury Park, N.J.\n\nFirst ...",2012,"Dec 1, 2012\n\ncolumn: ANIMAL DOCTOR",B,Gannett Media Corp,"Asbury Park, N.J.",...,News,1316685534,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,48.0,1.000000,"['dogs', 'dog', 'owners', 'humans', 'drug', 'a...","Your dog's eyes must be constantly monitored, ...",0.75,"(0.6, 0.8]"
1469,Dry corneas can lead to blindness,Any rapid blinking (blepharospasm) or rubbing ...,Dear Dr. Fox: Our 4-year-old schnauzer has had...,Cats; Eyes & eyesight,"The Washington Post;

***
This topic is more about the healthy aging of pets
***

#### Topic 97

In [42]:
df[df['topic']==97]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
10,"Meal sites, service for Tuesday canceled",The closure includes the Senior Meal sites in ...,CRIS Healthy-Aging Center is closing all Senio...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.3",2010,"Dec 21, 2010",NaN,News Gazette,"Champaign, Ill.",...,News,839009193,https://www.proquest. com/globalnewsfed/newspa...,2016-08-21,97.0,0.071765,"['united', 'award', 'campaign', 'valley', 'ymc...","In addition, CRIS Rural Transit is canceling a...",0.78125,"(0.6, 0.8]"
71,United Way seeks to recover shortfall,"Programs assist with rental assistance, food a...",Marshfield-Marshfield Area United Way's campai...,Savings banks\n\nCompany / organization: Name:...,"Marshfield News Herald; Marshfield, Wis.",2010,"Nov 19, 2010",MNH,"Ogden Newspapers, Inc.","Marshfield, Wis.",...,News,807414869,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,97.0,1.000000,"['united', 'award', 'campaign', 'valley', 'ymc...",Companies that increased contributions by 50 p...,0.78125,"(0.6, 0.8]"
116,'The need is great',"""We don't have enough (volunteers), either,"" s...","URBANA - Once or twice a month, Marilyn Boasti...",NaN,"News Gazette; Champaign, Ill.\n\nFirst page: A.8",2010,"Oct 24, 2010",NaN,News Gazette,"Champaign, Ill.",...,News,762486933,https://www.proquest.com/globalnewsfed/new spa...,2016-08-21,97.0,0.045488,"['united', 'award', 'campaign', 'valley', 'ymc...",Provena Faith in Action will hold two orientat...,0.78125,"(0.6, 0.8]"
154,Faith in Action program seeking more volunteers,Volunteers help people 55 and older who need h...,URBANA - Volunteers are needed to help older a...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.2",2010,"Oct 8, 2010",NaN,News Gazette,"Champaign, Ill.",...,News,759595128,https://www.pro quest.com/globalnewsfed/newspa...,2016-08-21,97.0,0.029714,"['united', 'award', 'campaign', 'valley', 'ymc...",The Provena Center for Healthy Aging's Faith i...,0.78125,"(0.6, 0.8]"
671,"Severn River Middle students raise $4,200 for ...","""With this $4,200, we can pay people's utiliti...",The cafeteria at Severn River Middle School fi...,NaN,Capital; Annapolis\n\nFirst page: A.7,2011,"Dec 19, 2011",NaN,"Tribune Publishing Company, LLC",Annapolis,...,News,912001793,https://www.proquest.com/globalnewsfed/newspap...,2016-10-22,97.0,0.029641,"['united', 'award', 'campaign', 'valley', 'ymc...","""Last year, we collected enough toys on the Sa...",0.78125,"(0.6, 0.8]"
769,Need greater than expected in northern Champai...,"According to [Amy Marchant], CRIS defines nort...",RANTOUL - When CRIS Rural Mass Transit Distric...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.1",2011,"Oct 24, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,902214594,htt ps://www.proquest.com/globalnewsfed/newspa...,2016-08-21,97.0,0.105113,"['united', 'award', 'campaign', 'valley', 'ymc...",Rides within the north part of the county cost...,0.78125,"(0.6, 0.8]"
912,"2011-12 campaign sets off toward $700,000 goal","United Way officials hope to raise $700,000 th...","DANVILLE - Last year, thousands of students in...",NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.1",2011,"Sep 7, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,893379657,https://www. proquest.com/globalnewsfed/newspa...,2016-08-21,97.0,1.000000,"['united', 'award', 'campaign', 'valley', 'ymc...",n Project SuccessI 21st Century Community Lear...,0.78125,"(0.6, 0.8]"
1186,Volunteers needed for Faith in Action program,Volunteers provide transportation to doctorIs ...,CHAMPAIGN - Volunteers are needed to help olde...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.3",2011,"Apr 22, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,866956828,https://www.proq uest.com/globalnewsfed/newspa...,2016-08-21,97.0,0.025730,"['united', '

In [43]:
df.loc[119,'Full text']

'Sharing the belief that it\'s important for baby boomers and older adults to explore conservative measures in addition to surgical procedures when treating back pain, orthopedist John Hicks and Pardee Rehab and Wellness Center\'s manual physical therapist David Gerrer are the keynote speakers at the Thursday, Oct. 28, Great Life Series workshop, "Living Well with Your Aching Back." The lunch and learn program will be held from 11:15 a.m. to 2 p.m. in the Jamison Conference Room at Pardee Hospital. A complimentary lunch will be provided by Chef Starr to Go, and free valet parking is available at Pardee\'s front entrance off of 800 N. Justice St. "Back surgery is pretty gratifying," said Hicks of Blue Ridge Bone and Joint. "It can provide astonishing pain relief from symptoms of nerve compression such as the rip roaring pain down the leg that is triggered by nerve pressure from a bulging disc in the back." As effective as surgical procedures are, however, Hicks encourages patients not t

***
This topic is more about volunteering for the elderly than actually about healthy ageing.
***

In [44]:
df = remove_topic(df,48,[])

The orig length:11000 -> cropped: 10952


#### Topic 113

In [45]:
df[df['topic']==113]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
75,Eye health fair scheduled,URBANA - Provena Covenant Medical Center will ...,URBANA - Provena Covenant Medical Center will ...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.4",2010,"Nov 17, 2010",NaN,News Gazette,"Champaign, Ill.",...,News,815397225,https://www.proquest.com/globalnewsf ed/newspa...,2016-08-21,113.0,0.082936,"['classes', 'fair', 'class', 'mondays', 'pm', ...","There will also be information booths, refresh...",0.655172,"(0.6, 0.8]"
138,Fall Health Fair coming next week,The event is sponsored by Prowers Medical Cent...,The Healthy Aging Fall Health Fair will take p...,Senior citizen centers,"The Lamar Ledger; Lamar, Colo.",2010,"Oct 14, 2010",Health,Prairie Mountain Media,"Lamar, Colo.",...,News,918645397,https://www.proquest.com/globalnewsfed/newspap...,2012-03-19,113.0,1.000000,"['classes', 'fair', 'class', 'mondays', 'pm', ...",Those attending are recommended to have an 8-1...,0.655172,"(0.6, 0.8]"
461,Active Adults Get Results at the Senior Health...,"Workshops will include Chair Yoga, Healthy Coo...",Reader Submitted BRIDGEWATER Residents are inv...,Aging; Older people; Community centers,"Courier - News; Bridgewater, N.J.",2010,"Apr 26, 2010",GETPUBLISHED,Gannett Media Corp,"Bridgewater, N.J.",...,News,193934129,https://www.proquest.com/globalnewsfed/newspap...,2017-11-03,113.0,0.092732,"['classes', 'fair', 'class', 'mondays', 'pm', ...",Vendor demonstrations will include blood press...,0.655172,"(0.6, 0.8]"
547,Health fair,None available.,Strathcona County women will have a chance to ...,Health services\n\nIdentifier / keyword: count...,Sherwood Park News; Sherwood Park\n\nFirst pag...,2010,"Mar 2, 2010",News,Postmedia Network Inc.,Sherwood Park,...,News,2177180798,https://www.proquest.com/globalnewsfed/newspap...,2019-02-08,113.0,0.093027,"['classes', 'fair', 'class', 'mondays', 'pm', ...","Edmonton's Rejuvenation Health Services, a cer...",0.655172,"(0.6, 0.8]"
632,Health calendar: SPECIAL,"AARP DRIVER SAFETY CLASSES for those over 50, ...","""CAMP COOL,"" a two-day multimedia, interactive...",Fees & charges; Digital photography; Workshops...,"Honolulu Advertiser; Honolulu, Hawaii",2010,"Jan 14, 2010",LIFE,Gannett Media Corp,"Honolulu, Hawaii",...,News,414953309,https://www.proquest.com/globalnewsfed/newspap...,2017-11-14,113.0,1.000000,"['classes', 'fair', 'class', 'mondays', 'pm', ...",Mammograms for uninsured and underinsured wome...,0.655172,"(0.6, 0.8]"
938,Summer WHAM returns Aug. 30,The event was created and organized by a group...,"Summer WHAM, a free seniors wellness event, re...",NaN,"The Review; Richmond, B.C.\n\nFirst page: 1",2011,"Aug 25, 2011",Community,"Torstar Syndication Services, a Division of To...","Richmond, B.C.",...,News,885299446,https://www.proquest.com/globalnewsfed/newspap...,2011-08-26,113.0,0.130959,"['classes', 'fair', 'class', 'mondays', 'pm', ...","In addition, there will be 20 display tables o...",0.655172,"(0.6, 0.8]"
1010,Annual Healthy Aging Summit event Friday,None available.,"Be it the economy, lack of health insurance or...",Diabetes; Older people; Diabetic retinopathy; ...,Record Searchlight; Redding\n\nFirst page: C.2,2011,"Jul 29, 2011",Life,Gannett Media Corp,Redding,...,News,2592962173,https://www.proquest.com/globalnewsfed/newspap...,2021-11-04,113.0,0.075008,"['classes', 'fair', 'class', 'mondays', 'pm', ...",This particular exam is free to those who qual...,0.655172,"(0.6, 0.8]"
1099,Hispanic fair aims at good health: 400 atten...,[...] the last time he had a checkup was at th...,"testing, education By Joseph Gerth jgerth@cour...",Diet; Aging; Bone density; Hispanic Americans,"Courier - Journal; Louisville, Ky.",2011,"Jun 12, 2011",News,Gannett Media Corp,"Louisville, Ky.",...,News,871554550,https://www.proq

In [46]:
df.loc[472,'Full text']

"Several local natives took part Monday in the grueling Boston Marathon, including 2006 Miss Southeastern Ohio Jenna Wilson. A native of Zanesville and contestant in the Miss Ohio scholarship pageant the past two years, Wilson, 23, completed the course in 5 hours, 40 minutes and 43 seconds, which put her at 22,242nd in the sprawling field. She was 9,311th among women and 4,865th in her age group. According to her blog, Wilson has been training for about six months and ran as part of the Tufts University team's Tufts Presidents Marathon Challenge. She had to raise $1,000 to support nutrition, medical and fitness programs at Tufts. Tufts is a leader in research on healthy aging, childhood obesity and in its work to eliminate famine. Wilson, who also competed in several tune-up races and the Miss Boston pageant, is an Emerson-Tufts grad student in the Health Communication program. The Zanesville High grad finished as fourth-runner-up in the Miss Ohio pageant in 2008 and third in 2009. Oth

***
This topic is more about the Healthy Aging Fairs being held and the advertisements for their services.
***

In [47]:
df = remove_topic(df,113,[])

The orig length:10952 -> cropped: 10923


#### Topic 122

In [48]:
df[df['topic']==122]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
256,Southeast Asian health ministers' meet in Thai...,None available.,"New Delhi, Aug. 30 -- Health ministers from 11...",NaN,The Hindustan Times; New Delhi,2010,"Aug 30, 2010\n\nDateline: New Delhi",NaN,HT Digital Streams Limited,New Delhi,...,WIRE FEED,748812701,https://www.proquest.com/globalnewsfed/newspap...,2018-02-26,122.0,0.059485,"['sustainable', 'protection', 'countries', 'po...",The health ministers' meeting are expected to ...,0.666667,"(0.6, 0.8]"
1971,The human dynamo behind WHO: Director-genera...,"'What is development if, at the end of the day...","If you passed Dr Margaret Chan on the street, ...",Cardiovascular disease; Antibiotics; Tobacco; ...,The Straits Times; Singapore,2012,"Mar 25, 2012",Think,SPH Media Limited,Singapore,...,News,940553189,https://www.proquest.com/globalnewsfed/newspap...,2012-04-25,122.0,0.058637,"['sustainable', 'protection', 'countries', 'po...","I can store up sleep. On weekends, I can sleep...",0.666667,"(0.6, 0.8]"
5239,Vietnam: Health issues to heat up APEC SOM 3 a...,None available.,Health officials from the Asia-Pacific Economi...,Working groups; Tuberculosis\n\nLocation: Asia...,Asia News Monitor; Bangkok,2017,"Aug 25, 2017",General News,Thai News Service Group,Bangkok,...,News,1931698337,https://www.proquest.com/globalnewsfed/newspap...,2024-11-26,122.0,1.000000,"['sustainable', 'protection', 'countries', 'po...","Besides, APEC officials will mull over other i...",0.666667,"(0.6, 0.8]"
5244,Asia: Producing a Healthy Asia Pacific by 2020...,None available.,The health of all residents in the Asia-Pacifi...,Working groups; Cooperation; Aging\n\nLocation...,Asia News Monitor; Bangkok,2017,"Aug 23, 2017",General News,Thai News Service Group,Bangkok,...,News,1930807388,https://www.proquest.com/globalnewsfed/newspap...,2024-11-26,122.0,0.060279,"['sustainable', 'protection', 'countries', 'po...","The chair of the working group, Rocío Cathia C...",0.666667,"(0.6, 0.8]"
5807,World: Helping ensure growth is shared by men ...,None available.,Challenge No country can achieve its potential...,Gender differences; Collaboration; Initiatives...,Asia News Monitor; Bangkok,2018,"Oct 3, 2018",General News,Thai News Service Group,Bangkok,...,News,2115449909,https://www.proquest.com/globalnewsfed/newspap...,2024-12-05,122.0,0.045248,"['sustainable', 'protection', 'countries', 'po...",Moving Forward The WBG is building further mom...,0.666667,"(0.6, 0.8]"
6525,ASEAN: Joint Statement of the 14th ASEAN Healt...,None available.,"WE, the Ministers of Health of ASEAN Member St...",Food safety; Collaboration; Nutrition; Disease...,Asia News Monitor; Bangkok,2019,"Sep 23, 2019",General News,Thai News Service Group,Bangkok,...,News,2295103530,https://www.proquest.com/globalnewsfed/newspap...,2024-11-24,122.0,0.225398,"['sustainable', 'protection', 'countries', 'po...",13. We recognise and appreciate the contributi...,0.666667,"(0.6, 0.8]"
6659,United Nations: Big Challenges Remain to Healt...,None available.,Following are UN Secretary-General Antonio Gut...,International conferences; Population; Young a...,Asia News Monitor; Bangkok,2019,"Jul 22, 2019",General News,Thai News Service Group,Bangkok,...,News,2260589328,https://www.proquest.com/globalnewsfed/newspap...,2024-11-25,122.0,1.000000,"['sustainable', 'protection', 'countries', 'po...","In November, the Governments of Kenya and Denm...",0.666667,"(0.6, 0.8]"
6660,United Nations: Despite Gains in Lowering Mate...,None available.,Following is UN Secretary-General António Gute...,Population growth; Trends; Maternal mortality;...,Asia News Monitor; Bangkok,2019,"Jul 22, 2019",General News,Thai News Service Group,Bangkok,...,News,2260585784,https://www.proquest.com/globalnewsfed/newspap...,2024-11-25,122.0,1.0000

In [49]:
df.loc[6575,'Full text']

'WORRIES about ageing are almost exclusively a concern for the ageing themselves. The young are too busy living fast to give a thought to what happens when the mind and body start slowing down. Even in Silicon Valley, where millions are being pumped into research aimed at turning back the clock, the leading practitioners and experts tend to be of a certain age themselves. With one exception. Laura Deming is only 25, yet her professional focus is entirely on finding ways to increase the healthy human lifespan of those in their 40s, 50s and beyond. Her goal isn\'t that pie-in-the-sky dream of life without any expiration date, but about how we can live longer, and grow old while still enjoying the physical and mental advantages of being young. \'Most people think of the body as something that breaks down over time and is unfixable, similar to a car rusting,\' she says. \'But there are some cars that have been around since 1910 because people are maintaining them and replacing the parts. O

***
This one is more about the politics on general health within Asia
***

In [50]:
df = remove_topic(df,122,[])

The orig length:10923 -> cropped: 10896


#### Topic 151

In [51]:
df[df['topic']==151]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
175,Your town in brief: Flu shots for seniors in M...,None available.,All communities Know someone 'locally inspirin...,NaN,"The Salt Lake Tribune; Salt Lake City, Utah",2010,"Sep 29, 2010",NaN,The Salt Lake Tribune,"Salt Lake City, Utah",...,News,755669970,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,151.0,0.143256,"['farmers', 'volunteers', 'market', 'victoria'...","Salt Lake County Ban hunger, not books - Salt ...",0.73913,"(0.6, 0.8]"
299,Farmers' Market helps seniors stay healthy,Without being able to purchase fresh fruits an...,CLEVELAND - Many Northeast Ohio senior residen...,Cities; Vegetables; Fruits; Older people; Nutr...,"Call & Post, All-Ohio edition; Cleveland, Ohio...",2010,"Jul 28-Aug 3, 2010",HEALTH,Call and Post,"Cleveland, Ohio",...,News,750539226,https://www.proquest.com/globalnewsfed/newspap...,2011-06-03,151.0,0.108688,"['farmers', 'volunteers', 'market', 'victoria'...",The program starts in the month of June and ru...,0.73913,"(0.6, 0.8]"
475,Volunteer opportunities abound,Salt Lake County Aging Services Healthy Aging ...,Here are some ways provided by the 2-1-1 Infor...,NaN,"Deseret News; Salt Lake City, Utah\n\nFirst pa...",2010,"May 2, 2010",Local,Deseret Digital Media,"Salt Lake City, Utah",...,News,230760848,https://www.proquest.com/globalnewsfed/newspap...,2011-10-27,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...",Head Start Salt Lake CAP -- Classroom voluntee...,0.73913,"(0.6, 0.8]"
573,At your service,"Activities Unlimited of Cowichan Valley, non-p...",VOLUNTEER OPPORTUNITIES Saanich Emergency Soci...,Volunteers; Yoga; Spirituality; Mens health; M...,"Times - Colonist; Victoria, B.C.\n\nFirst page...",2010,"Feb 14, 2010",Life,Postmedia Network Inc.,"Victoria, B.C.",...,News,348405545,https://www.proquest.com/globalnewsfed/newspap...,2023-07-11,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...","Peace Vigil. Wednesdays at the cenotaph, Belle...",0.73913,"(0.6, 0.8]"
619,Volunteers are needed in varied areas,Maliheh Free Clinic -- Volunteer interpreters ...,Here are ways to share time and talents with o...,NaN,"Deseret News; Salt Lake City, Utah\n\nFirst pa...",2010,"Jan 24, 2010",Local,Deseret Digital Media,"Salt Lake City, Utah",...,News,351807716,https://www.proquest.com/globalnewsfed/newspap...,2011-10-23,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...",Volunteers must understand medical terminology...,0.73913,"(0.6, 0.8]"
620,Volunteer corner,Salt Lake County Aging Services Healthy Aging ...,Dial 2-1-1 for the volunteer information desk ...,Volunteers,"The Salt Lake Tribune; Salt Lake City, Utah",2010,"Mar 31, 2010",NaN,The Salt Lake Tribune,"Salt Lake City, Utah",...,News,280584134,https://www.proquest.com/globalnewsfed/newspap...,2017-11-06,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...",Salt Lake CAP Head Start - Volunteers are need...,0.73913,"(0.6, 0.8]"
1068,Asheville-area business briefs: Construction...,None available.,NEWLAND - North Carolina developer O2 Energies...,NaN,"Asheville Citizen - Times; Asheville, N.C.",2011,"Jun 26, 2011",LIVING,Gannett Media Corp,"Asheville, N.C.",...,News,873665752,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...","For more information, contact the county manag...",0.73913,"(0.6, 0.8]"
1069,BRIEFCASE,None available.,Briefcase Construction begins on solar farm NE...,NaN,"Asheville Citizen - Times; Asheville, N.C.\n\n...",2011,"Jun 26, 2011",News,Gannett Media Corp,"Asheville, N.C.",...,News,1471190578,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,151.0,1.000000,"['farmers', 'volunteers', 'market', 'victoria'...",Best Buy donates Wii Fit

***
Amongst these topics, there are valuable articles such as 308,5385 but others are about farmers aging and the agriculture downfall as well as volunteers.
***

In [52]:
df = remove_topic(df,151,[308,5385])

The orig length:10896 -> cropped: 10873


#### Topic 174

In [53]:
df[df['topic']==174]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
306,Articles for Growing Older from non-profit org...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Articles for Growing Older from non-profit org...,NaN,"Winnipeg Free Press; Winnipeg, Man.",2010,"Apr 7, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,578687712,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...",Centrally located in Osborne Village. Informat...,0.789474,"(0.6, 0.8]"
374,Bangor senior center plans gardening talk,BANGOR - Seniors interested in learning more a...,BANGOR - Seniors interested in learning more a...,Senior citizen centers,"Bangor Daily News; Bangor, Me.\n\nFirst page: 3",2010,"Jun 11, 2010",B,Bangor Daily News,"Bangor, Me.",...,News,366709778,https://www.proquest.com/globalnewsfed/newspap...,2017-11-06,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...",Gardeners Gayle Crowley and Charlie Taylor wil...,0.789474,"(0.6, 0.8]"
385,What's going on for seniors,The fair will serve as a resource centre for s...,Healthy Aging Fair: Nurse Next Door Home Healt...,Aging,"North Shore News; North Vancouver, B.C.\n\nFir...",2010,"Jun 6, 2010",Seniors,Postmedia Network Inc.,"North Vancouver, B.C.",...,News,365901213,https://www.proquest.com/globalnewsfed/newspap...,2012-02-07,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...",Drop-in: $4.40. Info: 604-983-6362 or kshubert...,0.789474,"(0.6, 0.8]"
411,Senior Expo to feature Silver Frame Awards,Hammond Street Senior Center is offering a Hea...,Among the many activities that are part of the...,Garage sales; Health services; Senior citizen ...,"Bangor Daily News; Bangor, Me.\n\nFirst page: 4",2010,"May 18, 2010",B,Bangor Daily News,"Bangor, Me.",...,Column,288301606,https://www.proquest.com/globalnewsfed/newspap...,2017-11-05,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...","There is no admission to attend and ""hear old ...",0.789474,"(0.6, 0.8]"
779,Health fair features exhibits for every need,"VON Meals on Wheels, Bayshore Home Health Care...",Centres for Seniors Windsor presents The Senio...,Martial arts; Mental health; Home health care,"The Windsor Star; Windsor, Ont.\n\nFirst page:...",2011,"Oct 22, 2011",Life,Postmedia Network Inc.,"Windsor, Ont.",...,News,900344253,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...","Mail information to Natalie Taylor, 3448 Turne...",0.789474,"(0.6, 0.8]"
819,County digest,None available.,"50+EXPO Annual event, geared toward adults 50 ...",NaN,"The Baltimore Sun; Baltimore, Md.\n\nFirst pag...",2011,"Oct 9, 2011",LOCAL,"Tribune Publishing Company, LLC","Baltimore, Md.",...,News,912727289,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...","Daily fee of $26 includes snacks, lunch, coffe...",0.789474,"(0.6, 0.8]"
821,Happenings,"Sarah Nixon-Jackel, an Older Adult Wellness Nu...",New organizations and groups changing their sc...,Farmers markets; Libraries; Biodiversity,"Saskatoon Sun; Saskatoon, Sask.\n\nFirst page: 11",2011,"Oct 9, 2011",News,Postmedia Network Inc.,"Saskatoon, Sask.",...,News,898126800,https://www.proquest.com/globalnewsfed/newspap...,2020-03-12,174.0,1.0,"['pm', 'st', 'tuesday', 'tickets', 'ave', 'fri...",For information call 652-6831 or 653-5339. AGL...,0.789474,"(0.6, 0.8]"
1021,North State in brief,None available.,New members sought for band The Shasta College...,Bands,Record Searchlight; Redding\n\nFirst page: B.1,2011,"Jul 23, 2011",Local,Gannett Media Corp,Redding,...,News,2593421124,https://www.proquest.com/globalnewsfed/newspap...,2021-11-05,174.0,1.0,"['pm', 'st', 'tuesday', 'ticket

***
These all are noise articles.
***

In [54]:
df = remove_topic(df,174,[])

The orig length:10873 -> cropped: 10854


#### Topic 177

In [55]:
df[df['topic']==177]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
33,[Diary ],"Enquiries: Centre for Public Management, 02616...",December December 7: Leader to Leader series J...,Politics; International relations; Editing; Cl...,"The Canberra Times; Canberra, A.C.T.\n\nFirst ...",2010,"Dec 7, 2010",NaN,Rural Press Pty Ltd Australian Community Media...,"Canberra, A.C.T.",...,News,1020454324,https://www.proquest.com/globalnewsfed/newspap...,2012-06-15,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...","Enquiries: Bayley and Associates, 0262825660 o...",0.684211,"(0.6, 0.8]"
109,Growing old no barrier to helping out,"NESTA, the National Endowment for Science, Tec...",People in their 50s and 60s are being challeng...,NaN,Evening News; Edinburgh (UK)\n\nFirst page: 10,2010,"Oct 27, 2010",NaN,National World,Edinburgh (UK),...,News,760040954,https://www.proquest.com /globalnewsfed/newspa...,2012-02-09,177.0,0.197877,"['productivity', 'australia', 'political', 'ho...","NESTA, the National Endowment for Science, Tec...",0.684211,"(0.6, 0.8]"
1449,"Good buy, old friend Small shopping strips hav...",A Monash University study confirms small strip...,SMALL shopping strips fill a crucial role for ...,Older people; Food; Shopping centers,"Sunday Herald - Sun; Melbourne, Vic.\n\nFirst ...",2012,"Nov 18, 2012",NEWS,Nationwide News Pty Ltd,"Melbourne, Vic.",...,News,1162942399,https://www.proquest.com/globalnewsfed/newspap...,2012-11-18,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...","There is also an Indian restaurant, organic fr...",0.684211,"(0.6, 0.8]"
1743,So that's the Origin of the dreaded b-word: ...,"Typical, bloody typical. In a week when the ca...","Typical, bloody typical. In a week when the ca...",Interest rates,"Sydney Morning Herald; Sydney, N.S.W.\n\nFirst...",2012,"Jul 7, 2012",NaN,Fairfax Digital,"Sydney, N.S.W.",...,News,1023729373,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...",Scientologists were denounced by a tweeting Ru...,0.684211,"(0.6, 0.8]"
2380,Rudd's Condong whistlestop,"After nearly an hour at Condong, Rudd slipped ...",THERE'S a lot to be said for a country where y...,Aging; Beer,"Daily News; Tweed Heads, N.S.W.\n\nFirst page: 6",2013,"Aug 24, 2013",NaN,Nationwide News Pty Ltd,"Tweed Heads, N.S.W.",...,News,1427015889,https://www.proquest.com/globalnewsfed/newspap...,2013-08-23,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...",Alas Steve never got to tend to the nation's m...,0.684211,"(0.6, 0.8]"
2909,Experts reveal eight ways to keep people healt...,Alamy Regular exercise can keep you fit and we...,ILL health in old age can be prevented if loca...,Older people; Public health; Age; Aging,Express (Online); London (UK),2014,"Nov 16, 2014",NaN,Express Newspapers PLC,London (UK),...,News,1625267269,https://www.proquest.com/newspapers/experts-re...,2014-11-17,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...","But it warned that ""it also has the potential ...",0.684211,"(0.6, 0.8]"
3909,Millions allocated for rural nurses,A $6.4 million commitment from the Abbott gove...,A $6.4 million commitment from the Abbott gove...,Health care industry; Chronic illnesses; Prima...,"Nyngan Observer; Nyngan, N.S.W.",2015,"Jul 7, 2015",NaN,Rural Press Pty Ltd Australian Community Media...,"Nyngan, N.S.W.",...,News,1694865226,https://www.proquest.com/globalnewsfed/newspap...,2015-07-08,177.0,1.000000,"['productivity', 'australia', 'political', 'ho...",'The government's commitment builds on APNA's ...,0.684211,"(0.6, 0.8]"
3986,HELP FOR ELDERLYON THEIR OWN: Campaign targe...,Town hall bosses across the region want to set...,LONELINESS and ill health among Greater Manche...,NaN,Manchester Evening News; Glasgow 

In [56]:
df = remove_topic(df,177,[])

The orig length:10854 -> cropped: 10835


### Topic rel_ratio 0.8 - 1

In [57]:
df[(df["rel_ratio"] > 0.8) & (df["rel_ratio"] <= 1)]['topic'].unique()

array([ 55.,  56.,  58., 116.,  -1.,   8.,  51.,  24.,  86.,  29.,  50.,
       107.,  40.,   4.,  23.,   5., 189.,  98.,  44.,  14.,   1., 105.,
       109., 187., 138.,  13., 110., 142.,  16.,  66., 170., 148.,  91.,
       215., 154.,  21.,  46., 137., 165.,  49.,   7., 159.,  11., 132.,
         2.,  95., 139., 163., 190.,  27.,  18., 103., 202.,  10.,  77.,
        41.,  39., 213.,  42.,  43., 191., 211., 153.,  96., 117.,  19.,
       218., 208.,  92., 205., 130.,  75.,  84.,  36., 166., 106., 150.,
       184., 220., 128., 209.,  76.,  79.,  89.,  88., 112., 173.,  63.,
       157., 197.,  52.,  80., 175.,  15.,  38.,  87.,  22.,  64.,  37.,
       186.,  31.,  57.,  69., 129., 101., 127.,  61.,  32., 133., 207.,
       162.,  35., 182., 146., 200., 156., 100., 124., 203., 192.,  73.,
       185., 206.,   3., 195., 102., 221.,  68.,  53.,  60., 145., 180.,
        71., 223.,  93., 167.,  90., 119., 161.,  78., 179., 183., 143.,
        99.,  65., 225., 224., 169., 144., 176., 19

In [58]:
subset = df[(df["rel_ratio"] > 0.8) & (df["rel_ratio"] <= 1)]
for topic, keywords in subset.groupby('topic')['topic_keywords'].unique().items():
    print(topic, "\n", keywords)

-1.0 
 ['[]']
1.0 
 ["['india', 'senior citizens', 'citizens', 'geriatric', 'elders', 'welfare', 'indian', 'government', 'programme', 'delhi', 'older persons', 'ministry', 'healthy ageing', 'facilities', 'organised', 'published ht', 'ht', 'permission', 'geriatric medicine', 'homes', 'old age', 'content', 'elderly people', 'policy', 'sector', 'occasion', 'programmes', 'world health', 'palliative', 'requirement']"]
2.0 
 ["['foods', 'vitamin', 'eat', 'eating', 'vegetables', 'fats', 'fish', 'vitamins', 'fruits', 'nuts', 'grains', 'nutrients', 'protein', 'beans', 'meat', 'sugar', 'dietary', 'fruits vegetables', 'fruit', 'eggs', 'mediterranean', 'diets', 'antioxidants', 'dairy', 'oil', 'intake', 'heart disease', 'rice', 'red', 'cholesterol']"]
3.0 
 ["['al', 'rehabilitation', 'geriatric', 'clinic', 'corporation', 'patient', 'national health', 'strategy', 'geriatrics', 'healthy ageing', 'integrated', 'longterm care', 'medical director', 'primary', 'syndigateinfo', 'programme', 'syndigate', '

#### Topic 8

In [59]:
show(df[df['topic']==8]['Title'])

Loading ITables v2.4.4 from the internet... (need help?)


In [60]:
df.loc[10806,'Full text']

'According to an ever-increasing army of biohackers who are taking a DIY approach to making their brains, bodies and heart function better, and longer, death may be optional. As a cardiologist, here’s my take and what I do myself Death may be optional. At least, that is according to an ever-increasing army of biohackers who are taking a DIY approach to making their brains, bodies and hearts function better, and longer, by “hacking” their biology. Devotees of biohacking use biology, science, personal and shared experience, and technology to optimise health, wellbeing and lifespan, often with lifestyle interventions but sometimes with self-experimentation. The list of “biohacks” is extensive and includes extreme diets, supplement use, juicing, red-light therapy, cold exposure in the form of ice baths and cryotherapy, nootropics (aka “smart drugs”), full-body scans and frequent blood tests. At the most extreme end, biohackers are even attempting to modify their bodies with DIY gene-editin

***
Advertisement topic
***

In [61]:
df = remove_topic(df,8,[])

The orig length:10835 -> cropped: 10730


#### Topic 14

In [62]:
show(df[df['topic']==14]['Title'])

Loading ITables v2.4.4 from the internet... (need help?)


In [63]:
df.loc[2942,'Full text']

'URBANA -- Plans for a $13.3 million Feed Technology Complex at the University of Illinois got a $3.5 million boost from the state Friday. Gov. Pat Quinn visited campus to announce that the state will contribute money from the Illinois Jobs Now program toward the project on the UI\'s South Farms, which will replace the 1927 feed mill at the corner of Fourth Street and St. Mary\'s Road. The new Feed Technology Complex is a public-private partnership involving the UI, state of Illinois, industry partners and private donors. Archer Daniels Midland Co. will donate $1.5 million for the project, hoping to work with the UI to "create new and innovative food and feed solutions," according to Brent Fenton, president of the company\'s animal feeds division. The facility will allow students and professors to process customized animal feeds and support research and educational programs in crop and animal sciences, nutrition and food science. It will be used to develop technologies for the manufact

***
This topic focuses mainly on innovation, technology, and while healthy aging is mentioned, it only comes in-between sentences, i.e. it is not the main topic of these articles.
***

In [64]:
df = remove_topic(df, 14, [])

The orig length:10730 -> cropped: 10650


#### Topic 38

In [65]:
df[df['topic']==38]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
436,Your thanks,[...] we would also like to thank all of the v...,"Monday May 10, 2010 Editor of the Reformer: Th...",Fund raising; Travel; Libraries,"The Brattleboro Reformer; Brattleboro, Vt.",2010,"May 10, 2010",Opinion,"New England Newspapers, Inc.","Brattleboro, Vt.",...,News,252163348,https://www. proquest.com/globalnewsfed/newspa...,2011-11-01,38.0,0.064874,"['pm', 'st', 'library', 'oct', 'road', '630', ...","The cards, phone calls, generosity and incredi...",0.890909,"(0.8, 1.0]"
524,Channel 12,None available.,QVO|| SZ.?@.^¢}U¢O{.AQVSR¥ZS.T}¢.[O¢QV.?B;@> :...,NaN,"Idaho State Journal; Pocatello, Idaho\n\nFirst...",2010,"Mar 14, 2010",TV Schedules,Idaho State Journal,"Pocatello, Idaho",...,News,238850550,https://www.proquest.com/globalnewsfed/newspap...,2017-11-10,38.0,0.183601,"['pm', 'st', 'library', 'oct', 'road', '630', ...",[O'QV. @> 4:00 pm: Loma Linda University Churc...,0.890909,"(0.8, 1.0]"
545,Across the Valley: News from every city: Coa...,"Model schools show academic excellence, respon...",Community forum on violence is tonight A commu...,Scholarships & fellowships; Councils; Students...,"The Desert Sun; Palm Springs, Calif.",2010,"Mar 3, 2010",Valley,Gannett Media Corp,"Palm Springs, Calif.",...,News,439863159,https://www.proquest.com/globalnewsfed/newspap...,2017-11-16,38.0,0.131336,"['pm', 'st', 'library', 'oct', 'road', '630', ...",The library will provide the tools to make pai...,0.890909,"(0.8, 1.0]"
613,Daily guide,Teresa Heinz Kerry on breast cancer; Taste of ...,"ON BOSTON.COM Chat 10 a.m. Talk ""American Idol...",Musicians & conductors; Earthquakes; Classical...,"Boston Globe; Boston, Mass.",2010,"Jan 27, 2010",Living Arts,"Boston Globe Media Partners, LLC","Boston, Mass.",...,News,405196815,https://www.proquest.com/globalnewsfed/newspap...,2017-11-10,38.0,0.120763,"['pm', 'st', 'library', 'oct', 'road', '630', ...",NR (1951) Terminator 2: Judgment Day 10:30 p.m...,0.890909,"(0.8, 1.0]"
639,tv talk,"DAYTIME Early Show (7 a.m., CBS/2) - Airport l...","DAYTIME Early Show (7 a.m., CBS/2) - Airport l...",Aging; Television programs,"Newsday, Combined editions; Long Island, N.Y.\...",2010,"Jan 12, 2010",EXPLORE LI,Newsday LLC,"Long Island, N.Y.",...,News,280377388,https://www.proquest.com/globalnewsfed/newspap...,2017-11-07,38.0,0.086457,"['pm', 'st', 'library', 'oct', 'road', '630', ...","DAYTIME Early Show (7 a.m., CBS/2) - Airport l...",0.890909,"(0.8, 1.0]"
718,GATHERING IN GRATITUDE: Ceremony to honor ve...,None available.,"FAIRGROUNDS More than 1,000 people, including ...",Governors; Memorial services; Emotions,"Kitsap Sun; Bremerton, Wash.\n\nFirst page: A.1",2011,"Nov 21, 2011",Local,Gannett Media Corp,"Bremerton, Wash.",...,News,2582724766,https://www.proquest.com/globalnewsfed/newspap...,2021-10-18,38.0,0.066113,"['pm', 'st', 'library', 'oct', 'road', '630', ...","It's the camaraderie, he said. I like the spea...",0.890909,"(0.8, 1.0]"
746,Advisory CNN Wire Outlook,(CNN) -- Supervising News Editor Maggie Leung ...,(CNN) -- Supervising News Editor Maggie Leung ...,NaN,"St. Joseph News - Press; St. Joseph, Mo.",2011,"Nov 4, 2011",NaN,St. Joseph News - Press,"St. Joseph, Mo.",...,News,902235525,https://www.proquest.com/globalnewsfed/newspap...,2011-11-05,38.0,0.118151,"['pm', 'st', 'library', 'oct', 'road', '630', ...",COMMENTARY-Risman-Sexual-Harassment In my mind...,0.890909,"(0.8, 1.0]"
801,Community Briefing,None available.,ACTON CHILDREN COPING WITH STRESS - The Acton-...,NaN,"Boston Globe; Boston, Mass.",2011,"Oct 16, 2011",West,"Boston Globe Media Partners, LLC","Boston, Mass.",...,News,898515285,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,38.0,1.000000,"['pm', 'st', 'library', 'oct', 'road', '630', ...",Registration is required; ca

In [66]:
df.loc[718,'Full text']

"FAIRGROUNDS More than 1,000 people, including the governor, gathered Friday for an emotional celebration of area veterans. Many had tears in their eyes or lumps in their throats at various times throughout the 90-minute event at Kitsap Pavilion. For many in the crowd, it was hard to avoid the intense emotions as two dozen people were recognized for losing family members to war. Those in attendance watched somberly when the symbol of a fallen soldier was formed with rifle, helmet, boots and dog tags; when bagpipes moaned; rifles fired in salute; and buglers sounded taps. Navy Petty Officer Robert Medal choked up while reciting the POW/MIA place setting remembrance service, a symbol of the thousands of POW/MIAs still unaccounted for. It's very emotional for me, he said. Every time I lose one of my military brothers or sisters, even though I may not have known them, knowing of the sacrifice they made for the country and their families, it's beyond belief. Chris Gregoire said her most dif

***
These all are noise topics
***

In [67]:
df = remove_topic(df,38,[])

The orig length:10650 -> cropped: 10595


#### Topic 87

In [68]:
df[df['topic']==87]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
449,CELEBRATE LIFE IN THE MIDST OF CONSTANT CHANGE,"""She's always been a little reserved,"" [Ulla P...",Kindred Spirits Animal Sanctuary hosts annual ...,NaN,"The Santa Fe New Mexican; Santa Fe, N.M.\n\nFi...",2010,"May 2, 2010",SCOOP,Santa Fe New Mexican,"Santa Fe, N.M.",...,News,231292605,https://www.proquest.com/globalnewsfed/newspap...,2017-11-02,87.0,1.000000,"['dogs', 'dog', 'owners', 'animal', 'animals',...","On May 9, veterinarian Mary Anne Schadler offe...",0.882353,"(0.8, 1.0]"
2629,A dog's life may keep you young [Scot Region ],The researchers from Glasgow Caledonian Univer...,SCIENTISTS are set to find out if owning a dog...,Dogs; Older people; Health sciences,Daily Mail; London (UK)\n\nFirst page: 29,2013,"Apr 23, 2013",News,"Solo Syndication, a division of Associated New...",London (UK),...,News,1335037430,https://www.proquest.com/globalnewsfed/newspap...,2013-04-23,87.0,0.317050,"['dogs', 'dog', 'owners', 'animal', 'animals',...",The research will concentrate on the 'incident...,0.882353,"(0.8, 1.0]"
3117,Fundraiser today at Animal Friends Humane Society,Falls are a major reason older adults need to ...,The Animal Friends Humane Society will celebra...,Older people; Assisted living facilities; Ther...,"Cincinnati Enquirer; Cincinnati, Ohio\n\nFirst...",2014,"Aug 9, 2014\n\ncolumn: Area news",S,Gannett Media Corp,"Cincinnati, Ohio",...,News,1552130975,https://www.proquest.com/newspapers/fundraiser...,2014-08-09,87.0,1.000000,"['dogs', 'dog', 'owners', 'animal', 'animals',...",The Racer also is credited for sparking a roll...,0.882353,"(0.8, 1.0]"
3854,The Birches celebrates the dog days of summer,"""This event was a great opportunity for everyo...","With a late-summer heat wave in full force, it...",NaN,"Daily Herald; Arlington Heights, Ill.",2015,"Jul 29, 2015",Submitted Content,Daily Herald,"Arlington Heights, Ill.",...,News,1700215353,https://www.proquest.com/globalnewsfed/newspap...,2017-11-22,87.0,0.382768,"['dogs', 'dog', 'owners', 'animal', 'animals',...",The Birches Assisted Living in Clarendon Hills...,0.882353,"(0.8, 1.0]"
4418,What have cats ever done for us?,[...]The Cats Protection charity summarises th...,They argue that cats' tendency to kill birds a...,Dogs; Cardiovascular disease; Older people; Ag...,The Garstang Courier; Garstang (UK),2016,"Oct 5, 2016",Offbeat,NLA Media,Garstang (UK),...,News,1826777897,https://www.proquest.com/globalnewsfed/newspap...,2016-10-07,87.0,1.000000,"['dogs', 'dog', 'owners', 'animal', 'animals',...",Please try again. By uploading your file you a...,0.882353,"(0.8, 1.0]"
4665,HEALTHY AGING: Getting older gracefully with a...,"Volunteers from Good Mews, Cobb County Animal ...","Junior Nesbitt of Smyrna found his Princess, a...",Dogs; Animal shelters; Humane societies; Older...,"The Atlanta Journal - Constitution; Atlanta, Ga.",2016,"Jun 4, 2016",LIVING & ARTS,"Atlanta Journal Constitution, LLC","Atlanta, Ga.",...,News,1793712153,https://www.proquest.com/globalnewsfed/newspap...,2017-11-23,87.0,1.000000,"['dogs', 'dog', 'owners', 'animal', 'animals',...",Small dogs with short legs and long bodies -- ...,0.882353,"(0.8, 1.0]"
4737,The Birches Assisted Living welcomes three the...,"Now that spring has sprung, The Birches' thera...",There are three new residents at The Birches A...,NaN,"Daily Herald; Arlington Heights, Ill.",2016,"Apr 30, 2016",Submitted Content,Daily Herald,"Arlington Heights, Ill.",...,News,1785849148,https://www.proquest.com/globalnewsfed/newspap...,2017-11-22,87.0,0.715474,"['dogs', 'dog', 'owners', 'animal', 'animals',...",The Birches Assisted Living in Clarendon Hills...,0.882353,"(0.8, 1.0]"
4938,THE POWER OF PETS Study shows benefits of dog ...,According to the Humane Society of the United ...,The black a

***
This topic is about animals helping people live longer, while only one is about aging of an animal itself.
***

In [69]:
df = df[~df.index.isin([10367])]

#### Topic 95

In [70]:
df[df['topic']==95]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
155,Beth Judea to show 'In the Family' film tonigh...,"Widowed support group: 7-8:30 p.m. Tuesday, Oc...",Items that focus on physical and mental health...,NaN,"Daily Herald; Arlington Heights, Ill.\n\nFirst...",2010,"Oct 7, 2010",NEIGHBOR,Daily Herald,"Arlington Heights, Ill.",...,News,759255201,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,95.0,0.109152,"['registration', 'support group', 'senior cent...",Call (847) 556-1777. Walkers club: The Arlingt...,0.9375,"(0.8, 1.0]"
247,Support Groups,AL-ANON ALCOHOLISM SUPPORT: An anonymous fello...,AL-ANON ALCOHOLISM SUPPORT: An anonymous fello...,NaN,"The Sun; Lowell, Mass.",2010,"Sep 15, 2010",Lifestyle,The Lowell Sun,"Lowell, Mass.",...,News,750927020,https://www.proque st.com/globalnewsfed/newspa...,2010-09-16,95.0,1.000000,"['registration', 'support group', 'senior cent...","Meets first and third Tues. of every month, 5:...",0.9375,"(0.8, 1.0]"
271,Area cyclists fight cancer,"Brian Finkele and Tammy Hanley, both of Worces...","Brian Finkele and Tammy Hanley, both of Worces...",Health insurance; Cancer; Health care; Accredi...,"Telegram & Gazette; Worcester, Mass.\n\nFirst ...",2010,"Aug 19, 2010\n\ncolumn: MEDICAL MEMOS",HEALTH,"GateHouse Media, Inc.","Worcester, Mass.",...,News,745936690,https://www.proquest.com/globalnewsfed/newspap...,2024-12-01,95.0,0.055975,"['registration', 'support group', 'senior cent...",UMass Memorial Health Care insurance counselor...,0.9375,"(0.8, 1.0]"
372,Health Briefs: Sacred Heart Health System of...,None available.,Sacred Heart Health System will provide free h...,NaN,"Pensacola News Journal; Pensacola, Fla.",2010,"Jun 13, 2010",Online Life,Gannett Media Corp,"Pensacola, Fla.",...,News,375164842,https://www.proquest.com/globaln ewsfed/newspa...,2017-11-03,95.0,1.000000,"['registration', 'support group', 'senior cent...",Blood donors will receive a free waffle coupon...,0.9375,"(0.8, 1.0]"
1013,Alexandria and Arlington health calendar,Sullivan gives a virtual tour of the surgical ...,"Thursday, July 28 Seniors walking program 8-9 ...",Aging; Cancer; Senior citizen centers; Cancer ...,"The Washington Post; Washington, D.C.",2011,"Jul 28, 2011",LOCAL LIVING,WP Company LLC d/b/a The Washington Post,"Washington, D.C.",...,News,879449461,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,95.0,1.000000,"['registration', 'support group', 'senior cent...",703-228-4721 or www.nvso.us. - Compiled by Ria...,0.9375,"(0.8, 1.0]"
1030,HEALTH DIGEST,None available.,Blood donations The Greater Chesapeake and Pot...,NaN,"The Baltimore Sun; Baltimore, Md.\n\nFirst pag...",2011,"Jul 10, 2011",LOCAL,"Tribune Publishing Company, LLC","Baltimore, Md.",...,News,876067516,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,95.0,1.000000,"['registration', 'support group', 'senior cent...",Women's group will be held from 11 a.m. to 1 p...,0.9375,"(0.8, 1.0]"
1082,Lifesaving lessons,"At Field Day , radio amateurs, also known as ""...",The Glen Burnie Volunteer Fire Department will...,Fires; Fire stations; Amateur radio; Aging,"Maryland Gazette; Glen Burnie, Md.\n\nFirst pa...",2011,"Jun 18, 2011",NaN,"Tribune Publishing Company, LLC","Glen Burnie, Md.",...,News,873368929,https://www.proquest.com/globalnewsfed/newspap...,2016-10-22,95.0,0.075362,"['registration', 'support group', 'senior cent...",The Amateur Radio Club of the National Electro...,0.9375,"(0.8, 1.0]"
1117,Asheville area health calendar,Learn about a cutting edge approach to address...,Health calendar The complete Health Calendar i...,Autism; Hospitals; Palliative care; Lutheran c...,"Asheville Citizen - Times; Asheville, N.C.",2011,"Jul 12, 2011",LIVING,Gannett Media Corp,"Asheville, N.C.",...,News,875927422,https://www.proquest.com/

***
These all are advertisements about different health programs
***

In [71]:
df = remove_topic(df,95,[])

The orig length:10594 -> cropped: 10562


#### Topic 107

In [72]:
df[df['topic']==107]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
28,Anissa V. Rivera: Physical activity showcased,"This session highlights ""Physical Activity Opp...",The ever-popular Lunch at the Library program ...,NaN,"San Gabriel Valley Tribune; West Covina, Calif.",2010,"Dec 9, 2010",Local,Los Angeles Newspaper Group,"West Covina, Calif.",...,News,816798283,https://www.proquest.com/globalnewsfed/newspap...,2010-12-10,107.0,0.172898,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",The character in the title is a sharp 15-year-...,0.833333,"(0.8, 1.0]"
217,BULLETIN BOARD,TODAY * Highland Hospital will host its 14th a...,TODAY * Highland Hospital will host its 14th a...,NaN,"The Charleston Gazette; Charleston, W.V.\n\nFi...",2010,"Sep 15, 2010",Life,Charleston Newspapers,"Charleston, W.V.",...,News,750919174,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,107.0,0.123405,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",Refreshments will be served. The headquarters ...,0.833333,"(0.8, 1.0]"
817,Focus on wellness draws crowd of 300,Somehow we ended up with a whole heap of servi...,"IT started out with just an idea, to celebrate...",NaN,Derwent Valley Gazette; New Norfolk\n\nFirst p...,2011,"Oct 12, 2011",NaN,Font Public Relations,New Norfolk,...,News,897122446,https://www.proquest.com/globalnewsfed/newspap...,2011-10-11,107.0,1.000000,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",We are either slow learners or full of ambitio...,0.833333,"(0.8, 1.0]"
908,9/11 Anniversary Good Time To Give Back,Did you know that there are currently more tha...,Today is the 10th anniversary of one of Americ...,NaN,"Republican & Herald; Pottsville, Pa.",2011,"Sep 8, 2011\n\nDateline: Pottsville, PA",News,Republican & Herald,"Pottsville, Pa.",...,News,888304971,https://www.proquest.com/globalnewsfed/newspap...,2011-09-11,107.0,0.032816,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",Skills: A 45-hour training is mandatory. Locat...,0.833333,"(0.8, 1.0]"
1279,Provena Covenant hosts gathering,Photos by Sandra Gorman/for The News-Gazette D...,"Dr. Abraham Kocheril, a Christie Clinic cardio...",NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.5",2011,"Feb 27, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,856629439,https://www.proquest.com/glob alnewsfed/newspa...,2016-08-21,107.0,1.000000,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",Photos by Sandra Gorman/for The News-Gazette T...,0.833333,"(0.8, 1.0]"
1403,Seniors Calendar,None available.,WALKING CLUB FOR SENIORS: 11 a.m. Thursdays; P...,NaN,"Daily Record; Morristown, N.J.",2012,"Dec 25, 2012",A,Gannett Media Corp,"Morristown, N.J.",...,News,1243240030,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,107.0,0.039208,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...","204 Flanders-Drakestown Road, Flanders . Mount...",0.833333,"(0.8, 1.0]"
1760,Morsels: 'Healthy Eating' for budget-minded,The Carolina Mountain Ribfest is accepting app...,'Healthy Eating' for budget-minded Blue Ridge ...,NaN,"Times News; Hendersonville, N.C.",2012,"Jun 27, 2012",NaN,Halifax Media Group,"Hendersonville, N.C.",...,News,1022302764,https://www.proquest.com/globalnewsfed/newspap...,2012-11-02,107.0,0.064109,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",One National Grand SPAM Champion in the adult ...,0.833333,"(0.8, 1.0]"
1778,Check out the Senior Games,"The games, organized by the York County Area A...","York, PA - The York County Senior Games will b...",Aging; Older people,"York Daily Record; York, Pa.",2012,"Jun 18, 2012",News,Gannett Media Corp,"York, Pa.",...,News,1020927991,https://www.proquest.com/globalnewsfed/newspap...,2012-06-19,107.0,1.000000,"['ave', 'senior center', 'bay', 'st', 'pm', 'g...",89-year-old Mary Louise Smith of Shrewsbury ey...,0.833333,"(0.8, 1.0]"
2828,Check

***
These are ad articles about health-related associations.
***

In [73]:
df = remove_topic(df,107,[])

The orig length:10562 -> cropped: 10532


#### Topic 109

In [74]:
df[df['topic']==109]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
60,DVD seminar provides health advice,This DVD-based seminar was developed by Dr. Wi...,Are you wondering how to better boost your ene...,NaN,"The Dispatch; Lexington, N.C.",2010,"Nov 23, 2010",NaN,Halifax Media Group,"Lexington, N.C.",...,News,808533472,https://www.proquest.com/globalnewsfed/newspap...,2012-10-12,109.0,1.000000,"['september', 'active aging', 'ymca', 'seminar...",The program will be held at 1 p.m. Dec. 2 at t...,0.866667,"(0.8, 1.0]"
216,Center for Active Generations to have events f...,"Monday, Sept. 20- noon- ""Changing the way we a...",The Center for Active Generations will be host...,Aging; Older people,"Argus Leader; Sioux Falls, S.D.",2010,"Sep 16, 2010",UPDATES,Gannett Media Corp,"Sioux Falls, S.D.",...,News,750887012,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,109.0,0.044571,"['september', 'active aging', 'ymca', 'seminar...",Some of the weeks highlighted events incude: M...,0.866667,"(0.8, 1.0]"
242,"University of Wisconsin-Fox Valley presents ""A...","Oct. 18: ""Physical and Fit for Life"" with Jane...",MENASHA -- Four programs for healthy aging is ...,Aging; Health care; Medicine; Research; Corpor...,"The Post - Crescent; Appleton, Wis.",2010,"Sep 4, 2010",APC,Gannett Media Corp,"Appleton, Wis.",...,News,749465550,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,109.0,1.000000,"['september', 'active aging', 'ymca', 'seminar...","Oct. 18: ""Physical and Fit for Life"" with Jane...",0.866667,"(0.8, 1.0]"
329,Series on health and aging offered at Salem Ho...,None available.,A free five-week class on healthy living while...,NaN,"Statesman Journal; Salem, Or.",2010,"Jul 10, 2010",UPDATE,Gannett Media Corp,"Salem, Or.",...,News,607075383,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,109.0,0.087312,"['september', 'active aging', 'ymca', 'seminar...","SE, Building D, on the first floor. Registrati...",0.866667,"(0.8, 1.0]"
544,Healthy aging topic of USC Alumni Club mixer,None available.,The USC Alumni Club of the Desert mixer on Thu...,NaN,"The Desert Sun; Palm Springs, Calif.",2010,"Mar 3, 2010",LIFESTYLES,Gannett Media Corp,"Palm Springs, Calif.",...,News,439865787,https://www.proquest.com/globalnewsfed/newspap...,2017-11-16,109.0,1.000000,"['september', 'active aging', 'ymca', 'seminar...",The mixer is scheduled from 5 to 7 p.m. the De...,0.866667,"(0.8, 1.0]"
574,Healthy aging in spotlight,"u Thursday, March 4, 9:30 a.m.-noon: The Power...",If you want to know how to stay healthy longer...,NaN,"Times News; Hendersonville, N.C.",2010,"Feb 13, 2010",NaN,Halifax Media Group,"Hendersonville, N.C.",...,News,435372051,https://www.proq uest.com/globalnewsfed/newspa...,2012-11-02,109.0,1.000000,"['september', 'active aging', 'ymca', 'seminar...",The cost is $22 per person. To register online...,0.866667,"(0.8, 1.0]"
706,DR. OZ TO VISIT MIDLAND PARK,TV host Dr.,TV host Dr. Mehmet Oz will visit the Northwest...,NaN,The Record; Hackensack\n\nFirst page: L.2,2011,"Nov 26, 2011",LOCAL,Gannett Media Corp,Hackensack,...,News,3188220239,https ://www.proquest.com/globalnewsfed/newspa...,2025-04-22,109.0,1.000000,"['september', 'active aging', 'ymca', 'seminar...",TV host Dr. Mehmet Oz will visit the Northwest...,0.866667,"(0.8, 1.0]"
854,"Healthy Fall Festival targets families, kids",The free festival will include workshops and d...,Two Surprise businesses that promote good heal...,Aging; Physical fitness; Festivals,"Arizona Republic; Phoenix, Ariz.",2011,"Sep 28, 2011",Surprise Republic 1,Gannett Media Corp,"Phoenix, Ariz.",...,News,894706476,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,109.0,0.034086,"['september', 'active aging', 'ymca', 'seminar...",** Healthy Fall Festival When: 8 a.m. to 4 p.m...,0.866667,"(0.8, 1

***
Ad topic
***

In [75]:
df = remove_topic(df,109,[])

The orig length:10532 -> cropped: 10502


#### Topic 129

In [76]:
df[df['topic']==129]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
566,FROM THE REGION,Police Chief Andrew J. Sluckis said the men go...,Auburn Pair rob man with knife at Auburn Mall ...,Older people; Stabbings; Motion picture festiv...,"Telegram & Gazette; Worcester, Mass.\n\nFirst ...",2010,"Feb 21, 2010\n\ncolumn: FROM THE REGION",LOCAL NEWS,"GateHouse Media, Inc.","Worcester, Mass.",...,News,269049246,https://www.proquest.com/globalnewsfed/newspap...,2024-12-01,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...",Tickets are $10 for the public; $7 for Fitchbu...,0.961538,"(0.8, 1.0]"
1046,Senior Citizen Activities,"Wednesday - Morning cards, 8:30 a.m.; Hometown...",Shamokin-Coal Township Sunday - Wii bowling an...,NaN,"The News - Item; Shamokin, Pa.",2011,"Jul 07, 2011\n\nDateline: Shamokin, PA",News,Sample News Group,"Shamokin, Pa.",...,News\n\nProQuestdocument ID: 875573649,NaN,https://www.proquest.com/globalnewsfed/newspap...,2016-08-21,129.0,0.156519,"['port', 'st', 'pm', 'ave', 'road', 'library',...","Thursday - Healthy Steps, 9:30 a.m.; line danc...",0.961538,"(0.8, 1.0]"
1095,DAILY CALENDAR,None available.,"Today Noon Used Book Sale, to 6 p.m. Knights o...",NaN,"Times Herald; Port Huron, Mich.",2011,"Apr 20, 2011",KIOSK,Gannett Media Corp,"Port Huron, Mich.",...,News,863800949,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...","Capac Library, 111 N Main St. 4:30 p.m. Interm...",0.961538,"(0.8, 1.0]"
1305,DAILY CALENDAR: Today,None available.,"10 a.m.Waffle Wednesday, to noon. Free waffles...",NaN,"Times Herald; Port Huron, Mich.",2011,"Feb 16, 2011",KIOSK,Gannett Media Corp,"Port Huron, Mich.",...,News,853863053,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...","2 p.m.Folk Concert: The Reminisers, Sanctuary ...",0.961538,"(0.8, 1.0]"
1828,IN BRIEF: Healthy Aging seminar on sleep is ...,None available.,St. Joseph Mercy Port Huron hospital is having...,NaN,"Times Herald; Port Huron, Mich.",2012,"May 21, 2012",A-Section,Gannett Media Corp,"Port Huron, Mich.",...,News,1019536299,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...",This will be the second time the German bands ...,0.961538,"(0.8, 1.0]"
1837,Daily calendar: Special Events,"Capac Library, 111 N. Main St. St. Clair Count...","6th Annual Lilac Festival, 10 a.m.-5 p.m. May ...",Cancer; Contractors; Meetings; Bands; Medical ...,"Times Herald; Port Huron, Mich.",2012,"May 16, 2012",KIOSK,Gannett Media Corp,"Port Huron, Mich.",...,News,1019537978,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...",St. Clair County Community College Fine Arts T...,0.961538,"(0.8, 1.0]"
1907,DAILY CALENDAR: Today,None available.,"Garage and Bake Sale, 9 a.m.-5 p.m. Lunch from...",NaN,"Times Herald; Port Huron, Mich.",2012,"Apr 18, 2012",KIOSK,Gannett Media Corp,"Port Huron, Mich.",...,News,1012297740,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...","Registration requested. (810) 794-4911, Ext. 1...",0.961538,"(0.8, 1.0]"
1993,Senior Activities,"Monday - Morning cards and puzzles, 8:30 a.m.;...",Shamokin-Coal Township Monday - Morning cards ...,NaN,"The News - Item; Shamokin, Pa.",2012,"Mar 11, 2012\n\nDateline: Shamokin, PA",News,Sample News Group,"Shamokin, Pa.",...,News,927590997,https://www.proquest.com/globalnewsfed/newspap...,2016-08-21,129.0,1.000000,"['port', 'st', 'pm', 'ave', 'road', 'library',...","Center St. Patrick's Day party will be held, a...",0.961538,"(0.8, 1.0]"
2035,IN BRIEF: Marysville man cras

***
Ad topic
***

In [77]:
df = remove_topic(df,129,[])

The orig length:10502 -> cropped: 10476


#### Topic 139

In [78]:
df[df['topic']==139]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
159,Your town in brief: Tour 'mod' homes in WVC,None available.,West Valley City Mod home tour - Four homes in...,NaN,"The Salt Lake Tribune; Salt Lake City, Utah",2010,"Oct 6, 2010",NaN,The Salt Lake Tribune,"Salt Lake City, Utah",...,News,756776893,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...",Uninsured seniors are asked to donate $20 for ...,0.88,"(0.8, 1.0]"
207,Too Good to Miss,Ancaster-Dundas-Flamborough Ontario Early Year...,"For nonprofit, charity or fundraising events i...",NaN,"The Spectator; Hamilton, Ont.\n\nFirst page: GO.6",2010,"Sep 20, 2010",Go,"Torstar Syndication Services, a Division of To...","Hamilton, Ont.",...,News,7515 06663,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...","E., euchre, bid euchre, bingo, Wednesday, 1 p....",0.88,"(0.8, 1.0]"
373,Leadership program graduates 33 students,Following an initial two-day retreat to get to...,Thirty-three students graduated last month fro...,NaN,"Times News; Hendersonville, N.C.",2010,"Jun 12, 2010",NaN,Halifax Media Group,"Hendersonville, N.C.",...,News,375068722,https://www.proquest.com/globalnewsfed/newspap...,2012-11-02,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...","Lee Smith Utilities director, city of Henderso...",0.88,"(0.8, 1.0]"
467,Have your say,"James Shapiro's op-ed (Alas Poor Shakespeare, ...","Goodbye, Bard of Avon James Shapiro's op-ed (A...",NaN,"Winnipeg Free Press; Winnipeg, Man.\n\nFirst p...",2010,"Apr 21, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,580133637,https://www.proquest.com/g lobalnewsfed/newspa...,2017-11-17,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...",The Transportation Options Network for Seniors...,0.88,"(0.8, 1.0]"
627,Antique Show Benefits Woman's Club,"Advance sale tickets are $4, which will go to ...",The 47th Annual Antique Show will support the ...,NaN,"The Ledger; Lakeland, Fla.",2010,"Jan 18, 2010",NaN,Halifax Media Group,"Lakeland, Fla.",...,Feature,390158088,https://www.proquest.com/globalnewsfed/newspap...,2012-10-12,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...","[ Send your news regarding civic groups, socia...",0.88,"(0.8, 1.0]"
1370,Around the area,None available.,Clark county Nominees sought for Caring Hearts...,Public health; Nominations; Task forces; Older...,"Columbian; Vancouver, Wash.\n\nFirst page: C.3",2011,"Jan 7, 2011",Clark County,Columbian Publishing Company,"Vancouver, Wash.",...,News,2581187884,https://www.proquest.com/globalnewsfed/newspap...,2021-10-13,139.0,0.078537,"['vancouver', 'sept', 'st', 'hills', 'county',...",Building 17. The task force has to update its ...,0.88,"(0.8, 1.0]"
2018,The Pi Omega Chapter of Omega Psi Phi,"""Ever since the beginning, there's always been...","Since its inception, the Pi Omega Chapter of O...",Fraternities & sororities; Community service; ...,"Afro - American, 5 Star edition; Baltimore, Md...",2012,"Feb 25-Mar 2, 2012",NaN,Afro - American Company of Baltimore City,"Baltimore, Md.",...,News\n\nDocument feature: Photographs,928112028,https://www.proquest.com/globalnewsfed/newspap...,2012-03-15,139.0,1.000000,"['vancouver', 'sept', 'st', 'hills', 'county',...","Nearly eight months later, the Baltimore City ...",0.88,"(0.8, 1.0]"
2131,Your Thanks,"Morningside Shelter, Bonnyvale Environmental E...",Editor of the Reformer: From Nov. 8 through No...,Nonprofit organizations; Volunteers; Donations...,"The Brattleboro Reformer; Brattleboro, Vt.",2013,"Dec 16, 2013",Opinion,"New England Newspapers, Inc.","Brattleboro, Vt.",...,News,1468422445,https://www.proquest.com/globalnews

In [79]:
df = remove_topic(df,139,[])

The orig length:10476 -> cropped: 10452


#### Topic 154

In [80]:
df[df['topic']==154]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
103,People & Changes,MARK BOSTICK has been elected as chairman of t...,MARK BOSTICK has been elected as chairman of t...,NaN,"The Ledger; Lakeland, Fla.",2010,"Nov 1, 2010",NEWS,Halifax Media Group,"Lakeland, Fla.",...,News,761401304,https://www.proquest.com/globalnewsfed/newspap...,2012-10-12,154.0,1.000000,"['named', 'vice president', 'vice', 'president...","[ Send information for People & Changes, On th...",1.0,"(0.8, 1.0]"
129,Companies Associations/nonprofits Communicatio...,ITT of McLean named Kenneth W. Hunzeker vice p...,Strategic Enterprise Solutions of Reston named...,Appointments & personnel changes; Intellectual...,"The Washington Post; Washington, D.C.",2010,"Oct 18, 2010\n\nDateline: A",A-SECTION,WP Company LLC d/b/a The Washington Post,"Washington, D.C.",...,News,758828779,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,154.0,1.000000,"['named', 'vice president', 'vice', 'president...","Vorys, Sater, Seymour and Pease of the Distric...",1.0,"(0.8, 1.0]"
267,APPOINTMENTS,The National Council on Aging of the District ...,"Gannett of McLean named Travis Fore, former se...",Appointments & personnel changes; Natural prod...,"The Washington Post; Washington, D.C.",2010,"Aug 23, 2010",A SECTION,WP Company LLC d/b/a The Washington Post,"Washington, D.C.",...,News,746398166,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,154.0,1.000000,"['named', 'vice president', 'vice', 'president...",Andrews Kurth of the District named Eric R. Ma...,1.0,"(0.8, 1.0]"
721,Business People Twin Cities,"ADVOCACY Nancy Kielhofner, a former Allina Hos...","ADVOCACY Nancy Kielhofner, a former Allina Hos...",Executives; Appointments & personnel changes; ...,"Saint Paul Pioneer Press; Saint Paul, Minn.",2011,"Nov 19, 2011",Business People,St Paul Pioneer Press,"Saint Paul, Minn.",...,News,905005578,https://www.proquest.com/globalnewsfed/newspap...,2011-11-20,154.0,1.000000,"['named', 'vice president', 'vice', 'president...",Complete Nutrition is a nutritional supplement...,1.0,"(0.8, 1.0]"
2080,Business & professional briefs,"Accounting firm promotes three. Wendy Hendon, ...",New jobs Heiser named to bank post. Ryan E. He...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: E.3",2012,"Jan 15, 2012",NaN,News Gazette,"Champaign, Ill.",...,News,919209617,https://www.proquest.com/globalnewsfed/newspap...,2016-08-21,154.0,1.000000,"['named', 'vice president', 'vice', 'president...",Items may be emailed to dodson@news-gazette.co...,1.0,"(0.8, 1.0]"
2109,Who's News: Claerbout inducted into industry...,"Wendee Hauch, office manager and chiropractic ...","Daven Claerbout, co-owner/executive vice presi...",Aging; Small business; Associations; Continuin...,"Press; Sheboygan, Wis.",2012,"Jan 1, 2012",SHE,Sheboygan Press,"Sheboygan, Wis.",...,News,913480503,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,154.0,1.000000,"['named', 'vice president', 'vice', 'president...",Certified Professional Restoration adds projec...,1.0,"(0.8, 1.0]"
2335,DID YOU MISS?,None available.,BOARD DIRECTORS Advertised: September 7 Employ...,NaN,"The Advertiser; Adelaide, S. Aust.\n\nFirst pa...",2013,"Sep 14, 2013",Careerone,Nationwide News Pty Ltd,"Adelaide, S. Aust.",...,News,1432025133,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,154.0,0.052161,"['named', 'vice president', 'vice', 'president...",It wants to recruit up to three future-focused...,1.0,"(0.8, 1.0]"
2797,business people,"Bell State Bank & Trust, Golden Valley, named ...","moving up Liberty Diversified International, N...",Acquisitions & mergers; Executives\n\nLocation...,"Star Tribune; Minneapolis, Minn.\n\nFirst page...",2013,"Jan 28, 2013",BUSINESS,Star Tribune Media Company LLC,"Minneapolis, Minn.",...,News,12826542

***
Not a strongly related topic and also contains ads
***

In [81]:
df = remove_topic(df,154,[])

The orig length:10452 -> cropped: 10430


#### Topic 165

In [82]:
df[df['topic']==165]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
120,Large print books available at library,If you are physically unable to visit the bran...,The Windsor Public Library's philosophy is tha...,Libraries; Stress; Blood pressure,"The Windsor Star; Windsor, Ont.\n\nFirst page:...",2010,"Oct 22, 2010",News,Postmedia Network Inc.,"Windsor, Ont.",...,News,759711796,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...","130. Mail information to Natalie Taylor, 3448 ...",0.9,"(0.8, 1.0]"
1811,Fridge News; A Weekly Look At What's Happening...,The event goes Fridays from 10-11:30 a.m. It's...,TO GET YOUR ANNOUNCEMENT IN THE FRIDGE NEWS Ca...,NaN,"The Taber Times; Taber, Alta.\n\nFirst page: A.6",2012,"May 30, 2012",News,Postmedia Network Inc.,"Taber, Alta.",...,News,1018202590,https://www.proquest.com/globalnewsfed/newspap...,2016-09-17,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...",GENERAL INFORMATION L.T. Westlake Parent Advis...,0.9,"(0.8, 1.0]"
1822,Local briefs: Center for aging to hold open ...,None available.,RHINEBECK - The Center for Healthy Aging at No...,NaN,"The Poughkeepsie Journal; Poughkeepsie, N.Y.",2012,"May 25, 2012",Local News,Gannett Media Corp,"Poughkeepsie, N.Y.",...,News,1015792740,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...","For more information, call 845-889-4745 Ext. 1...",0.9,"(0.8, 1.0]"
2048,Church offers interesting Friday night date,Kathleen Shatt is on hiatus. [...] she returns...,The Women of John Wesley United Methodist Chur...,Dance; Bone density; Lutheran churches,"Maryland Gazette; Glen Burnie, Md.\n\nFirst pa...",2012,"Feb 11, 2012",NaN,"Tribune Publishing Company, LLC","Glen Burnie, Md.",...,News,921228207,https://www.proquest.com/globalnewsfed/news pa...,2016-10-22,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...","Registration is not required. For details, cal...",0.9,"(0.8, 1.0]"
2228,SFU Surrey: The month ahead,The SFU Surrey campus will host a Police Fair ...,The following is a look ahead at events to be ...,NaN,"The Leader; Surrey, B.C.\n\nFirst page: 1",2013,"Nov 5, 2013",Business,"Torstar Syndication Services, a Division of To...","Surrey, B.C.",...,News,1448416665,https://www.proquest.com/globalnewsfed/newspap...,2013-11-05,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...",Knowledge Creation @ SFU Surrey will feature a...,0.9,"(0.8, 1.0]"
2399,SENIORS,Seniors Crisis Counseling The Long Island Cris...,'Knit and Crochet for Charity' East Meadow: El...,Aging; Older people; Libraries; Nominations,"Newsday, Combined editions; Long Island, N.Y.\...",2013,"Aug 11, 2013",LI LIFE,Newsday LLC,"Long Island, N.Y.",...,News,1419107578,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...",'Savvy Senior' Nominations LI Association of G...,0.9,"(0.8, 1.0]"
2672,The week ahead,None available.,SUNDAY MARCH 24 The Ottawa Gatineau Internatio...,NaN,"The Ottawa Citizen; Ottawa, Ont.\n\nFirst page...",2013,"Mar 23, 2013",Life,Postmedia Network Inc.,"Ottawa, Ont.",...,News,1318989427,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,165.0,1.0,"['church', 'pm', 'club', 'welcome', 'march', '...",There will be eggs and chocolate bunnies to be...,0.9,"(0.8, 1.0]"
2939,Save the Dates,The event benefit the patients of the AIDS Act...,CRIME FIGHTING GALA: The Officer David M. Petz...,Charitable foundations; Art galleries & museums,"Morning Call; Allentown, Pa.\n\nFirst page: Go.7",2014,"Nov 2, 2014",Sunday_Go_Guide,"Tribune Publishing Company, LLC","Allentown, Pa.",...,News,1619427284,https://www.proquest.com/newspapers/save-dates...,2017-11-21,165.0,1.0,"['church'

***
Ad topic
***

In [83]:
df = remove_topic(df,165,[])

The orig length:10430 -> cropped: 10410


#### Topic 166

In [84]:
df[df['topic']==166]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
303,Annual Healthy Aging Fair a big draw for senio...,"""It offers us a chance to get checkups we othe...",HAYWARD -- Lucille Jackson considers any free ...,NaN,"Oakland Tribune; Oakland, Calif.",2010,"Jul 24, 2010",My Town,Bay Area News Group,"Oakland, Calif.",...,News,658343381,https://www.proquest.com/globalnewsfed/newspap...,2011-08-24,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...","""Free food and free gifts,"" Brown said. ""You c...",0.85,"(0.8, 1.0]"
588,NEIGHBORS BRIEFS: [Free program for senior c...,None available.,The Wellness Initiative for Senior Education P...,NaN,"Daily Journal; Vineland, N.J.",2010,"Feb 10, 2010",LOCAL,Gannett Media Corp,"Vineland, N.J.",...,News,1075124962,https://www.proquest.com/globalnewsfed/newspap...,2017-11-19,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...",Pottery classes at Clay College Cumberland Cou...,0.85,"(0.8, 1.0]"
628,Notices,Vancouver Coastal Health will be holding a com...,"Healthy Aging: The North Shore Chapter, Osteop...",Aging; Libraries; Young adults,"North Shore News; North Vancouver, B.C.\n\nFir...",2010,"Jan 17, 2010\n\ncolumn: Health Notes",Life,Postmedia Network Inc.,"North Vancouver, B.C.",...,News,361584312,https://www.proquest.com/globalnewsfed/newspap...,2012-02-08,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...","Info and registration: Lou, 604-924-2424, www....",0.85,"(0.8, 1.0]"
752,American Cancer Society representatives to speak,Current committees are focused on child obesit...,The Madison Community Health Consortium (MCHC)...,Advisors; Cancer; Meetings; Medical research\n...,"Asheville Citizen - Times; Asheville, N.C.\n\n...",2011,"Nov 2, 2011",A,Gannett Media Corp,"Asheville, N.C.",...,News,1471231656,https://www.proquest.com/globalnewsfed/newspap...,2017-11-21,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...",Some who can't attend meetings have contribute...,0.85,"(0.8, 1.0]"
932,Council offers useful programs,"Topics include mental health and well-being, m...","Have you ever wanted to learn a new skill, bec...",Aging; Older people,"Saskatoon Sun; Saskatoon, Sask.\n\nFirst page: 18",2011,"Aug 28, 2011\n\ncolumn: Prime Of Life",News,Postmedia Network Inc.,"Saskatoon, Sask.",...,Column,894374275,https://www.proquest.com/globalnewsfed/newspap...,2012-05-15,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...",Our blood pressure clinic continues to run on ...,0.85,"(0.8, 1.0]"
963,Healthy Aging Fair moves to Chabot College,"Jennifer Wu, of Fremont, gives Javier Romero, ...",HAYWARD -- There were some adjustments made fo...,NaN,"Oakland Tribune; Oakland, Calif.",2011,"Aug 13, 2011",My Town,Bay Area News Group,"Oakland, Calif.",...,News,883217 478,https://www.proquest.com/globalnewsfed/newspap...,2011-08-16,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...",The health fair included free health screening...,0.85,"(0.8, 1.0]"
1708,HEALTHY-AGING FAIR AT CHABOT COLLEGE,None available.,HAYWARD -- A Healthy Aging Fair will be held A...,NaN,"San Jose Mercury News; San Jose, Calif.\n\nFir...",2012,"Jul 30, 2012",Local,Bay Area News Group,"San Jose, Calif.",...,News,1033331251,https://www.proquest.com/globalnewsfed/newspap...,2012-08-14,166.0,1.0,"['fair', 'vancouver', 'summer', 'feb', 'shore'...",The countywide event provides a nutritious lun...,0.85,"(0.8, 1.0]"
1726,Summer WHAM offers wellness tips for seniors,"""Our goal with Summer WHAM is to give seniors ...",Discover the extensive communities and service...,NaN,"The Review; Richmond, B.C.\n\nFirst page: 12",2012,"Jul 16, 2012",Community,"Torstar Syndication Services, a Division of To...","Richmond, B.C.",...,News,1026539078,https://www.proquest.com/globalnewsfed/newspap...,2012-07-17,166.0,1.0,"['fair

In [85]:
df.loc[752,'Full text']

"The Madison Community Health Consortium (MCHC) quarterly meeting is scheduled for Wednesday Nov. 16 from noon to 1:30 p.m. at Cooperative Extension Auditorium, Marshall, NC. Noon to 12:30 p.m. is lunch and networking followed with the program & MCHC Committee Updates/Reports from 12:30 p.m. to 1:30 p.m. The program presentation will be from the American Cancer Society representatives about current programs, resources, and information about a newly funded cancer prevention initiative for Madison, Yancey, and Mitchell counties. Kathlene Stith, Community Health Advisor Manager, will provide information regarding the new program to help women access mammograms and other lifesaving screenings. The new Community Health Advisor (CHA) program will recruit and train women to educate and navigate their community members to screenings for breast, cervical and colorectal cancer. This program will start recruiting volunteers this November. Sarah Gayle, Community Manager, will share information abo

***
Ad topic about health events
***

In [86]:
df = remove_topic(df,166,[])

The orig length:10410 -> cropped: 10390


#### Topic 170

In [87]:
df[df['topic']==170]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
88,ACTIVITIES FOR 50-PLUS,The Healthy Aging Clinic is offering a communi...,Wheels To Meals starting up again The Wheels t...,Citizenship; Education; Meals; Older people; A...,"Daily Gleaner; Fredericton, N.B.\n\nFirst page...",2010,"Sep 21, 2010",Features,Postmedia Network Inc.,"Fredericton, N.B.",...,News,751827934,https://www.proquest.com/globalnewsfed/newspap...,2010-09-21,170.0,0.188212,"['noon', 'pm', 'class', 'sisters', 'begins', '...",UNB College of Extended Learning has No Limits...,1.0,"(0.8, 1.0]"
463,Healthy Aging Jamboree to be held Sunday at VF...,None available.,"Saturday April 24, 2010 BRATTLEBORO -- Brattle...",NaN,"The Brattleboro Reformer; Brattleboro, Vt.",2010,"Apr 24, 2010",Lifestyle,"New England Newspapers, Inc.","Brattleboro, Vt.",...,News,380329760,https://www.proquest.com/globalnewsfed/newspap...,2011-10-27,170.0,1.000000,"['noon', 'pm', 'class', 'sisters', 'begins', '...",Admission will be $5 and there will be dancing...,1.0,"(0.8, 1.0]"
508,FAITH IN THE CITY. A rock of devotion for near...,". Size and character of congregation: ""Approxi...",". Religious affiliation: The Episcopal Church,...",NaN,"New York Daily News; New York, N.Y.\n\nFirst p...",2010,"Mar 21, 2010",SUBURBAN,"Tribune Publishing Company, LLC","New York, N.Y.",...,News,306349576,https://www.proquest.com/globalnewsfed/newspap...,2017-11-02,170.0,1.000000,"['noon', 'pm', 'class', 'sisters', 'begins', '...",. Contact: (718) 885-1080; www.gracecityisland...,1.0,"(0.8, 1.0]"
1212,Moncton Headstart to host fundraising breakfas...,"Sue Stultz, New Brunswick minister for social ...","Sue Stultz, New Brunswick minister for social ...",Aging; Fund raising,"The Times - Transcript; Moncton, N.B.\n\nFirst...",2011,"Apr 9, 2011",Main,Postmedia Network Inc.,"Moncton, N.B.",...,News,861066321,https://www.proquest.com/globalnewsfed/newspap...,2011-04-09,170.0,1.000000,"['noon', 'pm', 'class', 'sisters', 'begins', '...",The event will begin at 8 a.m. with cost of ad...,1.0,"(0.8, 1.0]"
1447,Sisters in service Our Lady of Mercy nuns agin...,"""I know that God gives life, so I don't think ...",The OLM convent on James Island has a new supe...,NaN,"The Post and Courier; Charleston, S.C.\n\nFirs...",2012,"Nov 18, 2012",Features,The Post and Courier,"Charleston, S.C.",...,News,1163168249,https://www.proquest.com/globalnewsfed/newspap...,2012-11-18,170.0,1.000000,"['noon', 'pm', 'class', 'sisters', 'begins', '...","""Service never ends,"" Ritter adds. ""But we all...",1.0,"(0.8, 1.0]"
1546,Flamborough Seniors: Get on the bus,Flamborough Information and Community Services...,"Shelley Scott, executive director, Flamborough...",NaN,"Flamborough Review; Waterdown, Ont.",2012,"Oct 3, 2012",NaN,"Torstar Syndication Services, a Division of To...","Waterdown, Ont.",...,News,1082280027,https://www.proquest.com/glo balnewsfed/newspa...,2012-10-04,170.0,0.154009,"['noon', 'pm', 'class', 'sisters', 'begins', '...",You must have a registration card to attend th...,1.0,"(0.8, 1.0]"
2482,Meet & greet,The retirement community initiative has identi...,Reception set for newcomers Newcomers to this ...,Prostate cancer; Men,"The Windsor Star; Windsor, Ont.\n\nFirst page:...",2013,"Jun 29, 2013",Special Section,Postmedia Network Inc.,"Windsor, Ont.",...,Column,1372405106,https://www.proquest.com/globalnewsfed/newspap...,2017-11-20,170.0,1.000000,"['noon', 'pm', 'class', 'sisters', 'begins', '...",Email twhipp@windsorstar. com or call 519-255-...,1.0,"(0.8, 1.0]"
3474,Our Community: Full circle for volunteer's wor...,Because she grew up with a sister with a disab...,A young woman who came to Canada from Haiti as...,Colleges & universities; Music; Families & fam...,"Times - Colonist; Victoria, B.C.\n\nFirst page...",2014,"Feb 2, 2 014",Is

***
Health ad topic
***

In [88]:
df = remove_topic(df,170,[])

The orig length:10390 -> cropped: 10370


#### Topic 173

In [89]:
df[df['topic']==173]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
393,Health notes,LAKE WALES - As part of its Live Well! Communi...,LWMC will offer free screenings LAKE WALES - A...,NaN,"The Ledger; Lakeland, Fla.",2010,"Jan 10, 2010",NaN,Halifax Media Group,"Lakeland, Fla.",...,News,1177834866,https://www.proquest.com/globalnewsfed/newspap...,2012-11-21,173.0,1.000000,"['medical center', 'support group', 'valley', ...","The cost is $40, which includes a pet first ai...",1.0,"(0.8, 1.0]"
582,GET INVOLVED,RESOURCES American Heart Association Web site:...,RESOURCES American Heart Association Web site:...,Diabetes; Hospitals; Womens health; Cardiovasc...,"Morning Call; Allentown, Pa.\n\nFirst page: SP.1",2010,"Feb 11, 2010",Heart Health,"Tribune Publishing Company, LLC","Allentown, Pa.",...,News,393365150,https://www.proquest.com/globalnewsfed/newspap...,2017-11-06,173.0,1.000000,"['medical center', 'support group', 'valley', ...",225. WEIGHT LOSS New You Weight Loss Program W...,1.0,"(0.8, 1.0]"
644,Radiologist joins Fallon Clinic,Dr. Sharon J. Kuong has joined Fallon Clinic's...,Dr. Sharon J. Kuong has joined Fallon Clinic's...,Gastrointestinal surgery; Cancer therapies; Ag...,"Telegram & Gazette; Worcester, Mass.\n\nFirst ...",2010,"Jan 7, 2010\n\ncolumn: MEDICAL MEMOS",HEALTH & SCIENCE,"GateHouse Media, Inc.","Worcester, Mass.",...,News,269057504,https://www.proquest.com/globalnewsfed/newspap...,2024-12-01,173.0,1.000000,"['medical center', 'support group', 'valley', ...","For more information, call (888) 663-3688, ext...",1.0,"(0.8, 1.0]"
1160,Free self-care course offered,URBANA - If you or somebody you know has heart...,URBANA - If you or somebody you know has heart...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.3",2011,"May 9, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,869553264,https://www.proquest.com/globalne wsfed/newspa...,2016-08-21,173.0,0.104733,"['medical center', 'support group', 'valley', ...",Participants are asked to bring their medicati...,1.0,"(0.8, 1.0]"
1309,Heart Health Resources,"Heart-Pound ing stories of survival, recovery ...",This story appears in the Special Scetion: Hea...,Physical fitness; Families & family life; Hear...,"Morning Call; Allentown, Pa.\n\nFirst page: SP.1",2011,"Feb 13, 2011",Heart Health,"Tribune Publishing Company, LLC","Allentown, Pa.",...,News,851577019,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,173.0,1.000000,"['medical center', 'support group', 'valley', ...",* Clearing the Air: Smoking Cessation Support ...,1.0,"(0.8, 1.0]"
1368,THERE'S HELP FOR KEEPING NEW YEAR'S RESOLUTIONS,"A No, that causes withdrawal symptoms. You can...","The annual resolve to live better, healthier l...",NaN,"Daily Press; Newport News, Va.\n\nFirst page: A.6",2011,"Jan 7, 2011",Local,"Tribune Publishing Company, LLC","Newport News, Va.",...,News,822780769,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,173.0,1.000000,"['medical center', 'support group', 'valley', ...",... A two-minute walk every hour on the hour b...,1.0,"(0.8, 1.0]"
2283,"""Take Control of Your Health"" by Kimball Medic...",To help older adults and their caregivers mana...,Reader Submitted Center for Healthy Aging at K...,Aging; Hospitals; Chronic illnesses; Health fa...,"Asbury Park Press; Asbury Park, N.J.",2013,"Oct 7, 2013",Your News,Gannett Media Corp,"Asbury Park, N.J.",...,News,1467410993,https://www.proquest.com/globalnewsfed/newspap...,2017-11-21,173.0,1.000000,"['medical center', 'support group', 'valley', ...",In addition to offering a full range of innova...,1.0,"(0.8, 1.0]"
2298,Pittsfield: Chronic disease management program...,In partnership with Home Instead Senior Care a...,In partnership with Home Instead Senior Care a...,Older people; Aging,"The Berkshire Eagle; Pittsfield, Mass.",2013,"Sep 30

In [90]:
df.loc[644,'Full text']

"Dr. Sharon J. Kuong has joined Fallon Clinic's Radiology Department at the Gold Star Boulevard practice in Worcester. She received her medical degree from Tufts University School of Medicine in Boston. Dr. Kuong completed her residency program at both Boston Medical Center and Santa Clara Valley Medical Center in San Jose, Calif., and completed a fellowship program in body imaging (CT, MRI Ultrasound) at Tufts Medical Center, as well as a musculoskeletal fellowship program at the University of Washington Medical Center in Seattle, Wash. Dr. Kuong is certified by the American Board of Radiology. The Putnam Lions Club recently awarded Day Kimball Healthcare pediatrician Marc Cerrone its Humanitarian of the Year award, in recognition of his recent activities serving the health needs of children and families in impoverished areas of the world, along with his significant involvement in this community. Look Good ... Feel Better, a free American Cancer Society program that teaches cancer pat

***
Ad topic
***

In [91]:
df = remove_topic(df,173,[])

The orig length:10370 -> cropped: 10351


#### Topic 197

In [92]:
df[df['topic']==197]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
399,City Sidelines - May 27,Studio Babette Puppet Theatre presents Little ...,THURSDAY TOASTMASTERS Valley Town Toastmasters...,NaN,"Ancaster News; Ancaster, Ont.\n\nFirst page: 1",2010,"May 27, 2010",News,"Torstar Syndication Services, a Division of To...","Ancaster, Ont.",...,News,34 7095417,https://www.proquest.com/globalnewsfed/newspap...,2011-11-02,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...","Meetings are held Wednesday nights, 7:30 p. m....",1.0,"(0.8, 1.0]"
599,Little Black Dress travels to The Oaks,Eye on the prize Congratulations go to interna...,The Oaks Club recently welcomed Saks Fifth Ave...,Visual artists\n\nCompany / organization: Name...,"Sarasota Herald Tribune; Sarasota, Fla.\n\nFir...",2010,"Feb 2, 2010",B,Halifax Media Group,"Sarasota, Fla.",...,News,270870860,https://www.proquest.com/globalnewsfed/newspap...,2017-11-03,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...",For more information on the Hermitage Artist R...,1.0,"(0.8, 1.0]"
810,Community events,To list an event hosted or sponsored by a non-...,To list an event hosted or sponsored by a non-...,Aging; Garage sales,"Abbotsford Times; Abbotsford, B.C.\n\nFirst pa...",2011,"Oct 13, 2011",News,Postmedia Network Inc.,"Abbotsford, B.C.",...,News,898448300,https:// www.proquest.com/globalnewsfed/newspa...,2012-01-27,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...",Come and join a free breastfeeding class at Ab...,1.0,"(0.8, 1.0]"
1237,Vermilion libraries to kick off Money Smart We...,The nine public libraries in Vermilion County ...,The nine public libraries in Vermilion County ...,NaN,"News Gazette; Champaign, Ill.\n\nFirst page: B.3",2011,"Mar 25, 2011",NaN,News Gazette,"Champaign, Ill.",...,News,862906803,https: //www.proquest.com/globalnewsfed/newspa...,2016-08-21,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...","April 7: - 1 to 2 p.m. Fraud, Scam and Burglar...",1.0,"(0.8, 1.0]"
1601,Apple baking contest set for this weekend...[D...,"At 12:30 p.m., Billy Williams will auction off...",owen Apple baking contest set for this weekend...,Aging; Middle schools,"The Herald - Times; Bloomington, Ind.\n\nFirst...",2012,"Sep 14, 2012",NaN,"GateHouse Media, Inc.","Bloomington, Ind.",...,News,1039 316300,https://www.proquest.com/globalnewsfed/newspap...,2012-09-14,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...",Anyone with questions may call 330-7763. Monro...,1.0,"(0.8, 1.0]"
1803,COMMUNITY CALENDAR: THE WEEK OF JUNE 3,None available.,SUNDAY Port Jefferson Farmers market Local pro...,NaN,"Newsday, Combined editions; Long Island, N.Y.\...",2012,"Jun 3, 2012",EXPLORE LI,Newsday LLC,"Long Island, N.Y.",...,News,1018198857,https: //www.proquest.com/globalnewsfed/newspa...,2017-11-19,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...","WHEN|WHERE 10 a.m.-3 p.m., Trinity Evangelical...",1.0,"(0.8, 1.0]"
2051,What's On,"DECEMBER 1 ""Jake's Puzzle"" sponsoring a Tea fo...",MUSIC DECEMBER 1 & 2 A service of Carols and R...,Anglican churches; Christmas; Music,"Record; Sherbrooke, Que.\n\nFirst page: 12",2012,"Nov 30, 2012",Talk Of The Townships,Postmedia Network Inc.,"Sherbrooke, Que.",...,News,1221085218,https://www.proquest.com/globalnewsfed/newspap...,2012-12-03,197.0,1.0,"['december', 'pm', 'church', 'st', 'memorial',...",EXHIBITS UNTIL DECEMBER 16 Uplands is pleased ...,1.0,"(0.8, 1.0]"
2112,Guthrie Library Notes: Space needed for book sale,"""Digger Dozer Dumper,"" by Hope Vestergaard; ""D...",Many of us approach the New Year with a list o...,NaN,"The Evening Sun; Hanover, Pa.",2013,"Dec 31, 2013",Lifestyle,Gannett Media Corp,"Hanover, Pa.",...,News,1472538320,https://www.proquest.com/globalnewsfed/newspap...,2016-12-10,197.0,1.0,"['december', '

In [93]:
df.loc[5591,'Full text']

'Wednesdays and Mondays Faith Tabernacle Outreach Ministries Food Pantry: 11 am.-1 p.m., Faith Tabernacle lower level, 436 S. Jefferson St., Green Bay. Jan. 18 Home School Workshop: The Arctic: 9:30 a.m.-noon for grades 3-5, and 1-3:30 p.m. for grades 6-8, The NEW Zoo, Suamico. Drop-off program where students get hands-on learning opportunities. Learn about polar bears, glacial melting, habitat loss, predators and endangered species. Students should dress in layers and be ready for indoor and outdoor activities. $12 Zoo Pass members, $15 nonmembers. Reservations required. Visit newzoo.org. Peek at the Past: 12:30 p.m. at Aging & Disability Resource Center of Brown County, 300 S. Adams St., Green Bay. History of the national parks. ronaldpoister@new.rr.com. Overeaters Anonymous: 7 p.m., The Bridge, 2514 Jenny Lane, Green Bay. www.oa.org. Jan. 18-21 Blood Drives: 1-6 p.m. Jan. 19; 8 a.m.-noon Jan. 21, Green Bay Blood Donation Center, 2131 Deckner Ave., Green Bay. Noon-4 p.m. Jan. 18, Ras

***
Ad topic
***

In [94]:
df = remove_topic(df,197,[])

The orig length:10351 -> cropped: 10334


#### Topic 202

In [95]:
df[df['topic']==202]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,Document type,ProQuest document ID,Document URL,Last updated,topic,topic_prob,topic_keywords,last,rel_ratio,rel_bins
182,Manitoba Movers,has been appointed clinical consultant pharmac...,PEOPLE Guy Robbins and Lorrie Koroscil have be...,NaN,"Winnipeg Free Press; Winnipeg, Man.\n\nFirst p...",2010,"Sep 27, 2010",Business,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,News,754971632,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...",Dan Ryall of Fillmore Riley LLP has been appoi...,0.9375,"(0.8, 1.0]"
191,UCR may house healthy aging center,"White and G. Richard Olds, M.D., vice chancell...",The Desert Sun Palm Desert could be home to a ...,Physicians; Councils; Primary care\n\nLocation...,"The Desert Sun; Palm Springs, Calif.",2010,"Sep 24, 2010",VALLEY,Gannett Media Corp,"Palm Springs, Calif.",...,News,754125774,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,202.0,0.147776,"['grant', 'grants', 'received', 'diversity', '...",He told the council the university also aims t...,0.9375,"(0.8, 1.0]"
849,HUNTER COLLEGE'S SCHOOL MOVES INTO EAST HARLEM,"""Hunter is committed to this neighborhood. The...",HUNTER College's venerated School of Social Wo...,NaN,"New York Daily News; New York, N.Y.\n\nFirst p...",2011,"Sep 29, 2011",UPTOWN NEWS,"Tribune Publishing Company, LLC","New York, N.Y.",...,Ne ws,894757888,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...",Beshai is helping bakery trainees navigate the...,0.9375,"(0.8, 1.0]"
1355,"Future medical school taps valley physician, c...",The Desert Sun Coachella Valley physician and ...,The Desert Sun Coachella Valley physician and ...,Hospitals; Physicians\n\nLocation: Coachella V...,"The Desert Sun; Palm Springs, Calif.",2011,"Jan 13, 2011",VALLEY,Gannett Media Corp,"Palm Springs, Calif.",...,News,838205571,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...","UCR will pay Ruiz $65,000 annually for the par...",0.9375,"(0.8, 1.0]"
1668,Sac Town Native Comes to Los Angeles,From first hand experiences of cultural barrie...,Headnote to Continue Progressing Diversity in ...,Health care; Medical personnel; Workplace dive...,"Los Angeles Sentinel; Los Angeles, Calif.\n\nV...",2012,"Aug 16-Aug 22, 2012",FAMILY,Los Angeles Sentinel,"Los Angeles, Calif.",...,News,1039271637,https://www.proquest.com/globalnewsfed/newspap...,2017-04-15,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...",The Foundation prioritizes eight issues for fu...,0.9375,"(0.8, 1.0]"
1733,TCWF Recognizes Angela Minniefield for Commitm...,"Since its founding in 1992, TCWF has awarded 6...",Headnote Angela Minniefield Receives 2012 Cham...,Health care; Higher education; Awards & honors...,"Los Angeles Sentinel; Los Angeles, Calif.\n\nV...",2012,"Jul 12-Jul 18, 2012",LOCAL NEWS,Los Angeles Sentinel,"Los Angeles, Calif.",...,News\n\nDocument feature: Photographs,1030278985,https://www.proquest.com/globalnewsfed/newspap...,2017-04-15,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...",Please visit TCWF's website at CalWellness.org...,0.9375,"(0.8, 1.0]"
2502,TCWF Honors Health Education Leaders Committed...,Most recognized for her program Instant Recess...,The California Wellness Foundation (TCWF) hono...,Minority & ethnic groups; Public health; Healt...,"Los Angeles Sentinel; Los Angeles, Calif.\n\nV...",2013,"Jun 20-Jun 26, 2013",FAMILY Lifestyle HEALTH,Los Angeles Sentinel,"Los Angeles, Calif.",...,News\n\nDocument feature: Photographs,1399278155,https://www.proquest.com/globalnewsfed/newspap...,2017-04-15,202.0,1.000000,"['grant', 'grants', 'received', 'diversity', '...",Please visit TCWF's website at CalWellness.org...

***
The focus of the topic is not on either longevity or seniors
***

In [96]:
df = remove_topic(df,202,[])

The orig length:10334 -> cropped: 10318


<!-- structured-notebook -->
## Final Review Round
This last section revisits a later cleaned version (`df_v4`) and records the final targeted removals before writing the output used by subsequent notebooks.


### Topic cleaning v4

In [99]:
df = pd.read_csv(PROQUEST_PROCESSED_DIR / 'df_v4.csv')

In [100]:
for topic, keywords in df.groupby('topic')['topic_keywords'].unique().items():
    print(topic, "\n", keywords)

-1.0 
 ['[]']
0.0 
 ["['skin', 'hair', 'beauty', 'products', 'cream', 'lines', 'antiaging', 'vitamin', 'ingredients', 'product', 'sun', 'appearance', 'antiageing', 'oil', 'youthful', 'treatments', 'damage', 'fine', 'acid', 'pounds', 'water', 'antioxidants', 'signs', 'production', 'stem', 'cells', 'dry', 'contains', 'spots', 'routine']"]
1.0 
 ["['india', 'senior citizens', 'citizens', 'geriatric', 'indian', 'welfare', 'elders', 'delhi', 'elderly population', 'programme', 'care elderly', 'organised', 'ministry', 'facilities', 'elderly people', 'published ht', 'permission', 'ht', 'policy', 'older persons', 'geriatric medicine', 'homes', 'union', 'content', 'minister', 'sector', 'old age', 'demographic', 'economic', 'occasion']"]
2.0 
 ["['al', 'rehabilitation', 'corporation', 'geriatric', 'clinic', 'patient', 'national health', 'geriatrics', 'strategy', 'collaborating', 'longterm care', 'integrated', 'medical director', 'care older', 'tribune', 'primary', 'older persons', 'teams', 'progr

#### Topic 9

In [101]:
df[df['topic']==9]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,ISSN,Language of publication,Document type,ProQuest document ID,Document URL,Last updated,last,topic,topic_prob,topic_keywords
58,"too good to miss: For nonprofit, charity or ...",Alzheimer Society of Hamilton and Halton Care ...,All numbers are in 905 area code unless otherw...,NaN,"The Spectator; Hamilton, Ont.\n\nFirst page: G...",2010,"Nov 22, 2010",Go,"Torstar Syndication Services, a Division of To...","Hamilton, Ont.",...,11899417,English,News,807637429,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,"The McQuesten's Childhood Christmas, Nov. 20, ...",9.0,1.000000,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
139,Your town in brief: Winter symphony season sta...,None available.,Murray Come to the symphony - Murray's Symphon...,NaN,"The Salt Lake Tribune; Salt Lake City, Utah",2010,"Oct 6, 2010",NaN,The Salt Lake Tribune,"Salt Lake City, Utah",...,07463502,English,News,756774760,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,Murray Haul out your holly - The search is on ...,9.0,0.072010,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
152,Your town in brief: Fixing pit bulls for free ...,None available.,West Valley City Free neutering for pit bulls ...,NaN,"The Salt Lake Tribune; Salt Lake City, Utah",2010,"Sep 29, 2010",NaN,The Salt Lake Tribune,"Salt Lake City, Utah",...,07463502,English,News,755681980,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,The mall is located at 3601 S. 2700 West. Magn...,9.0,0.065165,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
187,Access Transcona offers the following. Call 93...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Access Transcona offers the following. Call 93...,NaN,"Winnipeg Free Press; Winnipeg, Man.\n\nFirst p...",2010,"Sep 17, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,08281785,English,News,751110174,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,The study includes physical assessment of the ...,9.0,1.000000,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
191,Access East offers the following. Call 938-500...,"Age & Opportunity Inc. ABC's of Fraud, seldom ...",Access East offers the following. Call 938-500...,NaN,"Winnipeg Free Press; Winnipeg, Man.",2010,"Sep 15, 2010",NaN,FP Canadian Newspapers Limited Partnership,"Winnipeg, Man.",...,08281785,English,News,750514928,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,The study includes physical assessment of the ...,9.0,1.000000,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8233,Things To Do,None available.,"Wednesday, March 9 ONLINE: Bingocize, 10-11 a....",Bible; Biblical studies; Aging,"Bangor Daily News; Bangor, Me.",2022,"Mar 7, 2022",NaN,Bangor Daily News,"Bangor, Me.",...,08928738\n\ne-ISSN: 26437457,English,News,2637161192,https://www.proquest.com/globalnewsfed/newspap...,2022-03-09,All others need to follow the 6 foot distance....,9.0,0.120718,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
8360,What's Happening,None available.,"Wednesday, Jan. 5 CARIBOU: Miss Erin's Picture...",Support groups; Public libraries; Food; Caregi...,"Bangor Daily News; Bangor, Me.",2022,"Jan 3, 2022",NaN,Bangor Daily News,"Bangor, Me.",...,08928738\n\ne-ISSN: 26437457,English,News,2616368550,https://www.proquest.com/globalnewsfed/newspap...,2022-01-14,"ONLINE: Caregiver Support Group, 12-1 p.m., ho...",9.0,0.129789,"['pm', 'st', 'wednesday', 'ave', 'church', 'cl..."
8361,Community Calendar,None available.,"Wednesday, Jan. 5 FORT FAIRFIELD: Senior commo...",Caregivers; Aging; Commodities\n\nBusiness ind...,"Bangor Daily News; Bangor, Me.",2022,"Jan 3, 2022",NaN,Bangor Daily News,"Bangor, Me.",...,08928738\n\ne-ISSN: 264 37457,English,News,2616368538,https://www.proquest.com/globalnewsfed/newspap...

***
Ad topic
***

In [107]:
df = remove_topic(df,9,[])

The orig length:10318 -> cropped: 10216


#### Topic 59

In [112]:
df[df['topic']==59]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,ISSN,Language of publication,Document type,ProQuest document ID,Document URL,Last updated,last,topic,topic_prob,topic_keywords
11,'Winter Blues' talk at Sylvia Ross Home,None available.,"BANGOR - Rosscare will hold an event called ""B...",NaN,"Bangor Daily News; Bangor, Me.\n\nFirst page: 3",2010,"Dec 21, 2010",B,Bangor Daily News,"Bangor, Me.",...,08928738\n\ne-ISSN: 26437457,English,News,818937716,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,"During the presentation, she will provide tips...",59.0,0.111724,"['pm', 'health education', 'road', 'meets', 'r..."
193,Asheville area health calendar,First meeting is Sept. 22; group will meet Wed...,The complete Health Calendar is posted online ...,Womens health; Shoulder; Health insurance; Phy...,"Asheville Citizen - Times; Asheville, N.C.",2010,"Sep 21, 2010",LIVING,Gannett Media Corp,"Asheville, N.C.",...,NaN,English,News,751848869,https://www.proquest.com/globalnewsfed/newspap...,2019-12-06,Call 452-8085 for a free consultation. WEIGHT ...,59.0,0.146876,"['pm', 'health education', 'road', 'meets', 'r..."
462,Health calendar: Services,Prenatal yoga is a wonderful way to honor your...,Divorce Recovery: 6 p.m. Tuesdays through Apri...,Hospitals; Social support; Yoga; Families & fa...,"Pensacola News Journal; Pensacola, Fla.",2010,"Mar 14, 2010",Online Life,Gan nett Media Corp,"Pensacola, Fla.",...,NaN,English,News,436240836,https://www.proquest.com/globalnewsfed/newspap...,2023-10-09,"Cokesbury United Methodist Church, 5725 N. Nin...",59.0,1.000000,"['pm', 'health education', 'road', 'meets', 'r..."
569,WEST COUNTY: County offers way to get fit in 2010,"The center's book club will meet to discuss ""S...",The county Recreation and Parks Department wil...,NaN,"Maryland Gazette; Glen Burnie, Md.\n\nFirst pa...",2010,"Jan 2, 2010",Community,"Tribune Publishing Company, LLC","Glen Burnie, Md.",...,NaN,English,News,374464115,https://www.proquest.com/globalnewsfed/newspap...,2016-10-22,Sign-up is required and there is a $10 fee per...,59.0,0.070219,"['pm', 'health education', 'road', 'meets', 'r..."
630,Health and fitness Send items to nbrcalendar@d...,Alzheimer's Caregivers Support Group: 1:30 to ...,Items that focus on physical and mental health...,NaN,"Daily Herald; Arlington Heights, Ill.\n\nFirst...",2011,"Nov 17, 2011",NEIGHBOR,Daily Herald,"Arlington Heights, Ill.",...,NaN,English,News,904414345,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,Total body conditioning class for people of al...,59.0,1.000000,"['pm', 'health education', 'road', 'meets', 'r..."
641,Free Health Screenings Available at Saturday G...,Speakers and their schedules are: Tent 1: 11 a...,"DAVENPORT | For the third year in a row, The P...",NaN,"The Ledger; Lakeland, Fla.",2011,"Nov 10, 2011",NEWS,Halifax Media Group,"Lakeland, Fla.",...,01630288,English,News,903160359,https://www.proquest.com/globalnewsfed/newspap...,2012-10-13,Nonprofit organizations include the Alzheimer'...,59.0,1.000000,"['pm', 'health education', 'road', 'meets', 'r..."
770,Local health news: Saint Mary's offers disease...,Saint Mary's Regional Medical Center is offeri...,Saint Mary's Regional Medical Center is offeri...,Aging; Blindness; Meditation,"Reno Gazette - Journal; Reno, Nev.",2011,"Sep 20, 2011",LIV,Gannett Media Corp,"Reno, Nev.",...,NaN,English,News,892602293,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,"After the meditation, enjoy conscious dance wi...",59.0,0.074019,"['pm', 'health education', 'road', 'meets', 'r..."
777,Health calendar,Monthly presentations by health care professio...,The complete Health Calendar is posted online ...,Medical research; Cancer therapies; Personal h...,"Asheville Citizen - Times; Asheville, N.C.",2011,"Feb 8, 2011",LIVING,Gannett Media Corp,"Asheville, N.C.",...,NaN,English,News,849673261,https://www.proquest.com/globalnewsfe

***
Ad topic
***

In [116]:
df = remove_topic(df,59,[])

The orig length:10216 -> cropped: 10171


#### Topic 72

In [117]:
df[df['topic']==72]

,Title,Abstract,Full text,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,...,ISSN,Language of publication,Document type,ProQuest document ID,Document URL,Last updated,last,topic,topic_prob,topic_keywords
291,INCORPORATIONS,"Noble Auto Body Shop, 06/02/10, Solomon Olubiy...",The following area businesses filed incorporat...,Condominiums; Aging,"Wisconsin State Journal; Madison, Wis.\n\nFirs...",2010,"Jul 8, 2010",BUSINESS,"Madison Newspapers, Inc.","Madison, Wis.",...,0749405X,English,News,596226566,https://www.proquest.com/globalnewsfed/newspap...,2017-11-17,The following area businesses filed incorporat...,72.0,1.000000,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
396,Arts Briefs,Based on several popular Dr. Seuss stories inc...,Henderson County YouTheatre Camp Registration ...,NaN,"Times News; Hendersonville, N.C.",2010,"May 3, 2010",NaN,Halifax Media Group,"Hendersonville, N.C.",...,10422323,English,News,250013111,https://www.proquest.com/glob alnewsfed/newspa...,2012-11-02,Dance admission is $20. Info: www.wcca.net or ...,72.0,1.000000,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
398,Henderson County,Based on several popular Dr. Seuss stories inc...,Registration is open for YouTheatre summer cam...,NaN,"Times News; Hendersonville, N.C.",2010,"May 2, 2010",NaN,Halifax Media Group,"Hendersonville, N.C.",...,10422323,English,News,231191186,https://www.proquest.com/globalnewsfed/newspap...,2012-11-02,"Dinner and dance tickets are $35, if purchased...",72.0,0.100961,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
635,In the Berkshires: Boys Club to host ice skati...,"At 5:30 p.m., in Building #1 at Mass MoCA, Wil...","Tuesday November 15, 2011 Pittsfield: Boys Clu...",Girls clubs; Garden clubs; Children & youth; S...,"The Berkshire Eagle; Pittsfield, Mass.",2011,"Nov 14, 2011",News,"New England Newspapers, Inc.","Pittsfield, Mass.",...,08958793,English,News,903813665,https://www.proquest.com/globalnewsfed/newspap...,2011-11-16,Registration: email chwalliance@gmail.com no l...,72.0,0.050045,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
675,Council candidates to drop in at Laurel Park T...,Four Seasons Rotary Club will join with Appleb...,Laurel Park--Council candidates to drop in at ...,NaN,"Times News; Hendersonville, N.C.",2011,"Oct 24, 2011",NaN,Halifax Media Group,"Hendersonville, N.C.",...,10422323,English,News,900431780,https://www.proquest.com/globalnewsfed/newspap...,2012-11-02,The Great Life Series is presented by Great Li...,72.0,0.143757,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
718,Community Calendar: Special Events Clubs Health,None available.,FUNDRAISER: Trinity Reformed United Church of ...,NaN,Intelligencer Journal / Lancaster New Era; Lan...,2011,"Oct 6, 2011\n\ncolumn: Community Calendar",B,LNP Media Group Inc.,"Lancaster, Pa.",...,NaN,English,News,896471927,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,Community Calendar runs as space is available....,72.0,0.034556,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
803,Senior centers plan special events in September,The Lexington Senior Center was established in...,The Davidson County Department of Senior Servi...,NaN,"The Dispatch; Lexington, N.C.",2011,"Sep 2, 2011",NaN,Halifax Media Group,"Lexington, N.C.",...,NaN,English,News,887259342,https://www.proquest.com/globalnewsfed/newspap...,2012-10-12,The Thomasville Senior Center opened its doors...,72.0,0.076241,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
863,Globe North Community briefing,None available.,Andover MORE DEER HUNTING - The Board of Selec...,NaN,"Boston Globe; Boston, Mass.",2011,"Aug 7, 2011",North,"Boston Globe Media Partners, LLC","Boston, Mass.",...,07431791,English,News,881494900,https://www.proquest.com/globalnewsfed/newspap...,2017-11-18,The hours are 7 to 11 p.m. on Friday and Satur...,72.0,0.195929,"['june', 'pm', 'county', 'road', 'monthly', 'r..."
1138,Washoe government meetings

***
Ad topic
***

In [120]:
df = remove_topic(df,72,[])

The orig length:10171 -> cropped: 10132


# Saving dataframe

In [129]:
df.to_csv(PROQUEST_PROCESSED_DIR / 'df_v4_cleaned.csv',index=False)